# MAKĀMIN | مكامن
## Graph-Based Fraud Detection & Investigation

**Team Members:**  
Nada Aljaafari - Lujain Alqarni -
Remas Almutairi - Alia AlGhamdi 


## Project Overview

Fraud is not always visible from a single suspicious transaction. Multiple transactions may share cards, addresses, email domains, or devices, creating relational patterns that are difficult to identify when transactions are analyzed independently.

This project uses the **IEEE-CIS Fraud Detection dataset** to investigate whether historical relational information can improve fraud detection beyond traditional transaction-level machine learning.

The project combines **XGBoost, Artificial Neural Networks (ANN), historical relational feature engineering, network analysis, and SHAP explainability**. A chronological validation strategy is used to reflect a realistic setting in which models learn from past transactions and evaluate future ones.

The final solution is integrated into **MAKĀMIN | مكامن**, an interactive fraud analytics and investigation dashboard that combines fraud risk scoring, prediction explanations, and historical transaction relationships.

## Project Objectives

The main objectives of this project are to:

1. Explore the characteristics of fraudulent and legitimate transactions.
2. Build a baseline fraud detection model using XGBoost.
3. Investigate relational patterns among fraudulent transactions using network analysis.
4. Incorporate temporal information to identify more meaningful transaction relationships.
5. Engineer historical relational features using only information available before each transaction and without using the fraud target.
6. Compare XGBoost and ANN models with and without the engineered historical features.
7. Evaluate model performance using metrics appropriate for an imbalanced fraud detection problem.
8. Explain the final model using SHAP at both global and individual transaction levels.
9. Integrate the final model, explanations, and historical relational analysis into the MAKĀMIN investigation dashboard.

# 1. Import Libraries

This section imports the libraries required for data manipulation, visualization, machine learning, network analysis, and model evaluation.

In [94]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Machine learning
from xgboost import XGBClassifier

# Model evaluation
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix
)

# Network analysis
import networkx as nx

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 100)

print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Load the Dataset

The IEEE-CIS Fraud Detection dataset contains transaction information and identity information stored in separate files. The training data will be used for model development and evaluation.

In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================

DATA_PATH = "/kaggle/input/competitions/ieee-fraud-detection"

train_transaction = pd.read_csv(
    f"{DATA_PATH}/train_transaction.csv"
)

train_identity = pd.read_csv(
    f"{DATA_PATH}/train_identity.csv"
)

print("Train Transaction Shape:", train_transaction.shape)
print("Train Identity Shape:", train_identity.shape)

# 3. Merge Transaction and Identity Data

The IEEE-CIS dataset stores transaction and identity information separately. Both datasets share `TransactionID`, which is used to combine the available transaction and identity information.

A left join is used so that all transactions are retained even when identity information is unavailable.

In [ ]:
# ============================================================
# 3. MERGE TRANSACTION AND IDENTITY DATA
# ============================================================

train = train_transaction.merge(
    train_identity,
    on='TransactionID',
    how='left'
)

print("Merged Training Shape:", train.shape)

# Verify the target variable
print("\nTarget column exists:", 'isFraud' in train.columns)

print("\nTarget distribution:")
print(train['isFraud'].value_counts())

fraud_rate = train['isFraud'].mean() * 100

print(f"\nFraud Rate: {fraud_rate:.2f}%")

# 4. Initial Data Understanding and Exploratory Data Analysis

Before building the fraud detection model, we examine the structure, data types, missing values, target imbalance, and selected transaction characteristics.

The purpose of this section is to understand the dataset and identify potential issues that may affect preprocessing and modeling.

In [ ]:
# ============================================================
# 4.1 DATASET STRUCTURE
# ============================================================

print("Number of Transactions:", train.shape[0])
print("Number of Features:", train.shape[1] - 1)

print("\nData Types:")
print(train.dtypes.value_counts())

print("\nDuplicate TransactionIDs:",
      train['TransactionID'].duplicated().sum())

In [ ]:
# ============================================================
# 4.2 MISSING VALUES
# ============================================================

missing_percentage = (
    train.isnull().mean() * 100
).sort_values(ascending=False)

print(
    "Columns with Missing Values:",
    (missing_percentage > 0).sum()
)

print("\nTop 15 Columns with Highest Missingness:")

display(
    missing_percentage.head(15).to_frame(
        name='Missing_Percentage'
    )
)

In [ ]:
# ============================================================
# 4.3 FRAUD CLASS DISTRIBUTION
# ============================================================

fraud_counts = train['isFraud'].value_counts()

plt.figure(figsize=(6, 4))

plt.bar(
    ['Non-Fraud', 'Fraud'],
    [fraud_counts[0], fraud_counts[1]]
)

plt.title('Distribution of Fraudulent and Non-Fraudulent Transactions')
plt.ylabel('Number of Transactions')

plt.show()

print(
    f"Non-Fraud: {fraud_counts[0]:,} "
    f"({fraud_counts[0] / len(train) * 100:.2f}%)"
)

print(
    f"Fraud: {fraud_counts[1]:,} "
    f"({fraud_counts[1] / len(train) * 100:.2f}%)"
)

In [ ]:
# ============================================================
# 4.4 TRANSACTION AMOUNT BY FRAUD STATUS
# ============================================================

amount_summary = (
    train.groupby('isFraud')['TransactionAmt']
    .agg(['count', 'mean', 'median', 'std'])
)

amount_summary.index = ['Non-Fraud', 'Fraud']

display(amount_summary.round(2))

## 4.5 Fraud Rate Across Selected Transaction Characteristics

Selected categorical variables are examined to determine whether fraud rates vary across different transaction, card, and device characteristics.

These comparisons are exploratory and do not by themselves establish that a feature causes fraudulent behavior.

In [ ]:
# ============================================================
# 4.5 FRAUD RATE BY SELECTED CATEGORICAL FEATURES
# ============================================================

categorical_features = [
    'ProductCD',
    'card4',
    'card6',
    'DeviceType'
]

for feature in categorical_features:
    
    print(f"\n{'=' * 60}")
    print(f"Fraud Analysis by {feature}")
    print('=' * 60)

    summary = (
        train.groupby(feature, dropna=False)['isFraud']
        .agg(['count', 'sum', 'mean'])
        .rename(columns={
            'count': 'Transactions',
            'sum': 'Fraud_Count',
            'mean': 'Fraud_Rate'
        })
    )

    summary['Fraud_Rate'] *= 100

    display(
        summary
        .sort_values('Fraud_Rate', ascending=False)
        .round(2)
    )

# 5. Chronological Train-Validation Split

Financial transaction data has a natural time order. A random train-validation split could allow information from later transactions to influence the evaluation of earlier transactions.

To better simulate a real-world fraud detection setting, the data is sorted by `TransactionDT`. The earliest 80% of transactions are used for training, while the latest 20% are reserved for validation.

All preprocessing decisions and model training will be based on the training period only.

In [ ]:
# ============================================================
# 5. CHRONOLOGICAL TRAIN-VALIDATION SPLIT
# ============================================================

# Sort transactions chronologically
train_sorted = (
    train
    .sort_values('TransactionDT')
    .reset_index(drop=True)
)

# 80/20 chronological split
split_index = int(len(train_sorted) * 0.80)

train_df = train_sorted.iloc[:split_index].copy()
valid_df = train_sorted.iloc[split_index:].copy()

print("Training Shape:", train_df.shape)
print("Validation Shape:", valid_df.shape)

print("\nTraining Fraud Rate:")
print(f"{train_df['isFraud'].mean() * 100:.2f}%")

print("\nValidation Fraud Rate:")
print(f"{valid_df['isFraud'].mean() * 100:.2f}%")

print("\nTransactionDT Ranges:")
print(
    "Training:",
    train_df['TransactionDT'].min(),
    "→",
    train_df['TransactionDT'].max()
)

print(
    "Validation:",
    valid_df['TransactionDT'].min(),
    "→",
    valid_df['TransactionDT'].max()
)

# Verify that the periods do not overlap
print(
    "\nChronological split verified:",
    train_df['TransactionDT'].max()
    < valid_df['TransactionDT'].min()
)

# 6. Data Preprocessing

The dataset contains a large number of numerical and categorical features with substantial missingness.

To reduce noise and computational cost, features with more than 95% missing values in the training period are removed. The threshold is determined using the training data only to avoid information leakage.

For the XGBoost model:

- Numerical missing values are retained as `NaN`, which XGBoost can handle natively.
- Categorical missing values are represented as `"Unknown"`.
- Categorical variables are converted to numerical category codes.
- Category mappings are learned from the training data and then applied to the validation data.
- `TransactionID` is excluded from the predictive feature set because it is an identifier.
- `isFraud` is separated as the target variable.

In [ ]:
# ============================================================
# 6.1 REMOVE EXTREMELY SPARSE FEATURES
# ============================================================

TARGET = 'isFraud'
ID_COL = 'TransactionID'

# Calculate missingness using TRAINING DATA ONLY
train_missing = train_df.isnull().mean()

# Remove features with more than 95% missing values
drop_cols = train_missing[
    train_missing > 0.95
].index.tolist()

# Never remove target or TransactionID here
drop_cols = [
    col for col in drop_cols
    if col not in [TARGET, ID_COL]
]

print("Columns removed:", len(drop_cols))

print("\nRemoved columns:")
print(drop_cols)

# Apply the SAME column removal to both periods
train_model = train_df.drop(columns=drop_cols).copy()
valid_model = valid_df.drop(columns=drop_cols).copy()

print("\nTraining shape after removal:", train_model.shape)
print("Validation shape after removal:", valid_model.shape)

In [ ]:
# ============================================================
# 6.2 SEPARATE FEATURES AND TARGET
# ============================================================

y_train = train_model[TARGET].copy()
y_valid = valid_model[TARGET].copy()

X_train = train_model.drop(
    columns=[TARGET, ID_COL]
).copy()

X_valid = valid_model.drop(
    columns=[TARGET, ID_COL]
).copy()

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

In [ ]:
# ============================================================
# 6.3 ENCODE CATEGORICAL FEATURES
# ============================================================

categorical_cols = X_train.select_dtypes(
    include=['object']
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    exclude=['object']
).columns.tolist()

print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numerical_cols))


for col in categorical_cols:

    # Missing category
    X_train[col] = X_train[col].fillna('Unknown').astype(str)
    X_valid[col] = X_valid[col].fillna('Unknown').astype(str)

    # Learn categories ONLY from training data
    train_categories = pd.Index(
        X_train[col].unique()
    )

    category_mapping = {
        category: code
        for code, category in enumerate(train_categories)
    }

    # Unknown validation categories become -1
    X_train[col] = (
        X_train[col]
        .map(category_mapping)
        .astype('int32')
    )

    X_valid[col] = (
        X_valid[col]
        .map(category_mapping)
        .fillna(-1)
        .astype('int32')
    )


print("\nCategorical encoding completed.")

print("Remaining object columns in training:",
      X_train.select_dtypes(include='object').shape[1])

print("Remaining object columns in validation:",
      X_valid.select_dtypes(include='object').shape[1])

In [ ]:
# ============================================================
# 6.4 VERIFY MODELING DATA
# ============================================================

print("Final Training Features:", X_train.shape)
print("Final Validation Features:", X_valid.shape)

print("\nMissing values:")
print("Training:", X_train.isnull().sum().sum())
print("Validation:", X_valid.isnull().sum().sum())

print("\nInfinite values:")
print(
    "Training:",
    np.isinf(
        X_train.select_dtypes(include=np.number)
    ).sum().sum()
)

print(
    "Validation:",
    np.isinf(
        X_valid.select_dtypes(include=np.number)
    ).sum().sum()
)

print("\nTarget distribution:")
print(y_train.value_counts())

# 7. Baseline XGBoost Model

A baseline XGBoost classifier is trained using the transaction and identity features after preprocessing.

Because fraudulent transactions represent only about 3.5% of the dataset, class imbalance is addressed using `scale_pos_weight`.

Model performance is evaluated primarily using Average Precision (AP / area under the precision-recall curve), together with ROC-AUC. These metrics are more informative than accuracy alone for an imbalanced fraud detection problem.

In [ ]:
# ============================================================
# 7.1 CALCULATE CLASS IMBALANCE WEIGHT
# ============================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Non-Fraud Transactions:", f"{negative_count:,}")
print("Fraud Transactions:", f"{positive_count:,}")
print("Scale Pos Weight:", round(scale_pos_weight, 2))

In [ ]:
# ============================================================
# 7.2 TRAIN BASELINE XGBOOST
# ============================================================

baseline_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(
    X_train,
    y_train
)

print("Baseline XGBoost training completed.")

In [ ]:
# ============================================================
# 7.3 EVALUATE BASELINE MODEL
# ============================================================

baseline_prob = baseline_model.predict_proba(X_valid)[:, 1]

baseline_ap = average_precision_score(
    y_valid,
    baseline_prob
)

baseline_roc = roc_auc_score(
    y_valid,
    baseline_prob
)

print("Baseline XGBoost Performance")
print("-" * 40)

print(f"Average Precision (AP): {baseline_ap:.4f}")
print(f"ROC-AUC:                {baseline_roc:.4f}")

print(f"\nRandom Baseline AP:     {y_valid.mean():.4f}")

# 8. Exploratory Fraud Network Analysis

After establishing the baseline fraud detection model, graph analysis is used to explore relationships among known fraudulent transactions.

Each fraudulent transaction is represented as a node. Transactions can be linked when they share selected identifying characteristics such as card and address information.

This fraud-only network is used exclusively for exploratory analysis. It is not used directly to generate predictive features because constructing model features from known fraud labels would introduce target leakage.

In [ ]:
# ============================================================
# 8.1 PREPARE FRAUD TRANSACTIONS FOR NETWORK ANALYSIS
# ============================================================

fraud_df = train[
    train['isFraud'] == 1
].copy()

network_cols = [
    'TransactionID',
    'TransactionDT',
    'TransactionAmt',
    'card1',
    'addr1',
    'P_emaildomain',
    'DeviceInfo'
]

fraud_network_data = fraud_df[
    network_cols
].copy()

print("Total Fraud Transactions:", len(fraud_network_data))

print("\nMissing Values in Network Features:")
print(
    fraud_network_data[
        ['card1', 'addr1', 'P_emaildomain', 'DeviceInfo']
    ].isnull().mean().mul(100).round(2)
)

## 8.2 Evaluation of Candidate Linking Features

Before constructing the fraud network, repeated values and combinations are examined to determine which attributes can meaningfully connect fraudulent transactions.

A useful linking feature should appear across multiple fraudulent transactions while maintaining sufficient data coverage.

In [ ]:
# ============================================================
# 8.2 EVALUATE CANDIDATE LINKING FEATURES
# ============================================================

candidate_features = [
    'card1',
    'addr1',
    'P_emaildomain',
    'DeviceInfo'
]

results = []

for feature in candidate_features:

    data = fraud_network_data[
        fraud_network_data[feature].notna()
    ]

    counts = data[feature].value_counts()

    repeated_values = counts[counts >= 2]

    linked_transactions = repeated_values.sum()

    results.append({
        'Feature': feature,
        'Fraud_Non_Missing': len(data),
        'Unique_Fraud_Values': data[feature].nunique(),
        'Repeated_Values': len(repeated_values),
        'Fraud_Transactions_Linked': linked_transactions,
        'Linked_Percentage':
            linked_transactions / len(fraud_network_data) * 100
    })


feature_link_summary = pd.DataFrame(results)

display(
    feature_link_summary.round(2)
)

## 8.3 Evaluation of Candidate Feature Combinations

Individual attributes may be shared by a large number of transactions and can therefore create overly broad network connections.

To define more specific transaction relationships, combinations of card, address, email, and device information are evaluated. The goal is to identify a linking rule that balances transaction coverage with relationship specificity.

In [ ]:
# ============================================================
# 8.3 EVALUATE CANDIDATE FEATURE COMBINATIONS
# ============================================================

candidate_combinations = [
    ['card1', 'addr1'],
    ['card1', 'P_emaildomain'],
    ['card1', 'DeviceInfo'],
    ['card1', 'addr1', 'P_emaildomain'],
    ['card1', 'addr1', 'DeviceInfo']
]

combination_results = []

for cols in candidate_combinations:

    # Only transactions with complete values for this combination
    temp = fraud_network_data.dropna(
        subset=cols
    )

    # Count transactions sharing each combination
    group_sizes = (
        temp.groupby(cols)
        .size()
    )

    # Groups containing at least two fraud transactions
    repeated_groups = group_sizes[
        group_sizes >= 2
    ]

    linked_transactions = repeated_groups.sum()

    combination_results.append({
        'Combination': ' + '.join(cols),
        'Complete_Fraud_Rows': len(temp),
        'Repeated_Groups': len(repeated_groups),
        'Linked_Fraud_Transactions': linked_transactions,
        'Linked_Percentage':
            linked_transactions / len(fraud_network_data) * 100
    })


combination_summary = pd.DataFrame(
    combination_results
)

display(
    combination_summary.round(2)
)

## 8.4 Time-Aware Fraud Network

A time-aware fraud network is constructed to identify more specific relationships among known fraudulent transactions.

Each fraudulent transaction with available `card1` and `addr1` information is represented as a node. Two transactions are connected when:

1. They share the same `card1`.
2. They share the same `addr1`.
3. They occur within 24 hours of each other.

The strength of each relationship is represented using an edge weight:

- **Weight 1:** Same card and address within 24 hours.
- **Weight 2:** Base relationship plus the same email domain or device information.
- **Weight 3:** Base relationship plus both the same email domain and device information.

This network is used for exploratory analysis only and does not use graph structure as a predictive feature.

In [ ]:
# ============================================================
# 8.4 BUILD TIME-AWARE FRAUD NETWORK
# ============================================================

TIME_WINDOW = 24 * 60 * 60   # 24 hours in seconds

# Only transactions with the required base identifiers
graph_data = (
    fraud_network_data
    .dropna(subset=['card1', 'addr1'])
    .sort_values('TransactionDT')
    .copy()
)

G_fraud = nx.Graph()

# Add transactions as nodes
for row in graph_data.itertuples():

    G_fraud.add_node(
        row.TransactionID,
        TransactionDT=row.TransactionDT,
        TransactionAmt=row.TransactionAmt,
        card1=row.card1,
        addr1=row.addr1,
        email=row.P_emaildomain,
        device=row.DeviceInfo
    )


# Group transactions by the required base relationship
for (_, _), group in graph_data.groupby(
    ['card1', 'addr1'],
    sort=False
):

    group = group.sort_values('TransactionDT')

    rows = list(group.itertuples())
    n = len(rows)

    # Sliding time window
    for i in range(n):

        j = i + 1

        while (
            j < n and
            rows[j].TransactionDT - rows[i].TransactionDT <= TIME_WINDOW
        ):

            r1 = rows[i]
            r2 = rows[j]

            weight = 1

            # Same email strengthens the relationship
            same_email = (
                pd.notna(r1.P_emaildomain)
                and pd.notna(r2.P_emaildomain)
                and r1.P_emaildomain == r2.P_emaildomain
            )

            # Same device strengthens the relationship
            same_device = (
                pd.notna(r1.DeviceInfo)
                and pd.notna(r2.DeviceInfo)
                and r1.DeviceInfo == r2.DeviceInfo
            )

            weight += int(same_email)
            weight += int(same_device)

            time_gap_hours = (
                r2.TransactionDT - r1.TransactionDT
            ) / 3600

            G_fraud.add_edge(
                r1.TransactionID,
                r2.TransactionID,
                weight=weight,
                time_gap_hours=time_gap_hours
            )

            j += 1


print("Time-Aware Fraud Network Created")
print("-" * 40)

print("Nodes:", f"{G_fraud.number_of_nodes():,}")
print("Edges:", f"{G_fraud.number_of_edges():,}")

components = list(nx.connected_components(G_fraud))

print("Connected Components:", f"{len(components):,}")

largest_component = max(
    (len(component) for component in components),
    default=0
)

print("Largest Component:", largest_component)

isolated_nodes = nx.number_of_isolates(G_fraud)

print("Isolated Transactions:", f"{isolated_nodes:,}")

## 8.5 Analysis of Network Relationship Strength

Edge weights represent the amount of shared information between connected fraudulent transactions.

- **Weight 1:** Same card and address within 24 hours.
- **Weight 2:** Same card and address within 24 hours, plus either the same email domain or device.
- **Weight 3:** Same card and address within 24 hours, plus both the same email domain and device.

Higher edge weights therefore represent stronger observed similarity between connected transactions.

In [ ]:
# ============================================================
# 8.5 EDGE WEIGHT DISTRIBUTION
# ============================================================

edge_weights = [
    data['weight']
    for _, _, data in G_fraud.edges(data=True)
]

edge_weight_summary = (
    pd.Series(edge_weights)
    .value_counts()
    .sort_index()
    .rename_axis('Edge_Weight')
    .reset_index(name='Number_of_Edges')
)

edge_weight_summary['Percentage'] = (
    edge_weight_summary['Number_of_Edges']
    / len(edge_weights)
    * 100
)

display(edge_weight_summary.round(2))

In [ ]:
# ============================================================
# 8.6 ANALYZE LARGEST CONNECTED COMPONENT
# ============================================================

components = list(nx.connected_components(G_fraud))

largest_nodes = max(
    components,
    key=len
)

largest_component_df = (
    fraud_network_data[
        fraud_network_data['TransactionID'].isin(largest_nodes)
    ]
    .sort_values('TransactionDT')
    .copy()
)

duration_hours = (
    largest_component_df['TransactionDT'].max()
    - largest_component_df['TransactionDT'].min()
) / 3600

print("Largest Component Summary")
print("-" * 40)

print("Transactions:", len(largest_component_df))

print(
    "Total Transaction Amount:",
    round(largest_component_df['TransactionAmt'].sum(), 2)
)

print(
    "Average Transaction Amount:",
    round(largest_component_df['TransactionAmt'].mean(), 2)
)

print(
    "Unique card1:",
    largest_component_df['card1'].nunique()
)

print(
    "Unique addr1:",
    largest_component_df['addr1'].nunique()
)

print(
    "Unique Email Domains:",
    largest_component_df['P_emaildomain'].nunique()
)

print(
    "Unique Devices:",
    largest_component_df['DeviceInfo'].nunique()
)

print(
    "Component Duration (Hours):",
    round(duration_hours, 2)
)

## 8.7 Visualization of the Largest Fraud Component

The largest connected component is visualized to illustrate the structure of relationships identified by the time-aware network.

Nodes represent fraudulent transactions, while edges represent shared card and address information within 24 hours. Edge width reflects relationship strength based on additional shared email or device information.

In [ ]:
# ============================================================
# 8.7 VISUALIZE LARGEST FRAUD COMPONENT
# ============================================================

largest_subgraph = G_fraud.subgraph(largest_nodes).copy()

plt.figure(figsize=(10, 7))

# Reproducible graph layout
pos = nx.spring_layout(
    largest_subgraph,
    seed=42
)

# Extract edge weights
weights = [
    largest_subgraph[u][v]['weight']
    for u, v in largest_subgraph.edges()
]

# Draw nodes
nx.draw_networkx_nodes(
    largest_subgraph,
    pos,
    node_size=180,
    alpha=0.8
)

# Draw weighted edges
nx.draw_networkx_edges(
    largest_subgraph,
    pos,
    width=weights,
    alpha=0.5
)

plt.title(
    "Largest Time-Aware Fraud Transaction Component\n"
    f"{len(largest_subgraph.nodes()):,} Transactions"
)

plt.axis('off')
plt.show()

# 9. Historical Network-Inspired Feature Engineering

The exploratory fraud network demonstrated that transactions can exhibit meaningful relationships through shared card, address, email, and device information.

To incorporate relational information into the predictive model without using the fraud target, historical network-inspired features are constructed for every transaction.

For each transaction, the features measure how many previous transactions within the preceding 24 hours shared selected identifiers.

The following historical features are created:

- `card1_prev_count_24h`: Previous transactions with the same card identifier.
- `addr1_prev_count_24h`: Previous transactions with the same address.
- `email_prev_count_24h`: Previous transactions with the same email domain.
- `device_prev_count_24h`: Previous transactions with the same device information.
- `card_addr_prev_count_24h`: Previous transactions sharing both card and address.

Only transactions occurring before the current transaction are counted. The fraud target (`isFraud`) is never used in feature construction.

In [ ]:
# ============================================================
# 9.1 CREATE HISTORICAL 24-HOUR RELATIONSHIP FEATURES
# ============================================================

from collections import defaultdict, deque

TIME_WINDOW = 24 * 60 * 60

# Work chronologically across the complete labeled dataset.
history_data = train_sorted[
    [
        'TransactionID',
        'TransactionDT',
        'card1',
        'addr1',
        'P_emaildomain',
        'DeviceInfo'
    ]
].copy()


def historical_count_24h(df, columns, feature_name):
    """
    Count previous transactions within 24 hours that share
    the specified identifier(s).

    Only past transactions are used.
    """

    histories = defaultdict(deque)
    counts = np.zeros(len(df), dtype=np.int32)

    values = df[columns].itertuples(
        index=False,
        name=None
    )

    times = df['TransactionDT'].to_numpy()

    for i, (key_values, current_time) in enumerate(
        zip(values, times)
    ):

        # Convert to tuple
        if not isinstance(key_values, tuple):
            key_values = (key_values,)

        # Missing identifier → no relationship feature
        if any(pd.isna(value) for value in key_values):
            counts[i] = 0
            continue

        key = key_values

        history = histories[key]

        # Remove transactions older than 24 hours
        while (
            history
            and current_time - history[0] > TIME_WINDOW
        ):
            history.popleft()

        # Number of PREVIOUS transactions still in window
        counts[i] = len(history)

        # Current transaction becomes history for future rows
        history.append(current_time)

    print(f"{feature_name} completed.")

    return counts


historical_features = pd.DataFrame({
    'TransactionID': history_data['TransactionID']
})


historical_features['card1_prev_count_24h'] = (
    historical_count_24h(
        history_data,
        ['card1'],
        'card1'
    )
)

historical_features['addr1_prev_count_24h'] = (
    historical_count_24h(
        history_data,
        ['addr1'],
        'addr1'
    )
)

historical_features['email_prev_count_24h'] = (
    historical_count_24h(
        history_data,
        ['P_emaildomain'],
        'email'
    )
)

historical_features['device_prev_count_24h'] = (
    historical_count_24h(
        history_data,
        ['DeviceInfo'],
        'device'
    )
)

historical_features['card_addr_prev_count_24h'] = (
    historical_count_24h(
        history_data,
        ['card1', 'addr1'],
        'card + address'
    )
)


print("\nHistorical features created:")
print(historical_features.shape)

display(historical_features.head())

### 9.1.1 Temporal Validation of Historical Features 

To verify that the historical relational features are leakage-safe, the dataset is checked for strict chronological ordering before model evaluation.

Each transaction's historical features are constructed using only transactions that occurred earlier in time within the preceding 24-hour window. The fraud target (`isFraud`) is not used during feature construction.

This validation confirms that:

- transactions are processed chronologically;
- the training period occurs before the validation period;
- historical features use only prior transaction activity;
- fraud labels are not used to generate relational features.

In [ ]:
# ============================================================
# 9.1.1 VERIFY TEMPORAL FEATURE CONSTRUCTION
# ============================================================

# Confirm that the full dataset is chronologically ordered
is_chronological = train_sorted["TransactionDT"].is_monotonic_increasing

# Compare training and validation time boundaries
last_train_time = train_df["TransactionDT"].max()
first_valid_time = valid_df["TransactionDT"].min()

print("Historical Feature Validation")
print("=" * 45)

print("Data sorted chronologically:", is_chronological)
print("Last training TransactionDT:", last_train_time)
print("First validation TransactionDT:", first_valid_time)

print(
    "Training period occurs before validation:",
    last_train_time < first_valid_time
)

print("\nFeature construction rule:")
print(
    "Each transaction uses only previous transactions "
    "within the preceding 24 hours."
)

print(
    "Fraud target used in historical features:",
    False
)

## 9.2 Historical Feature Distribution

The historical relationship features are examined before model integration to verify that they were generated correctly and to understand the frequency of repeated transaction identifiers within the preceding 24-hour window.

In [ ]:
# ============================================================
# 9.2 INSPECT HISTORICAL FEATURE DISTRIBUTIONS
# ============================================================

historical_cols = [
    'card1_prev_count_24h',
    'addr1_prev_count_24h',
    'email_prev_count_24h',
    'device_prev_count_24h',
    'card_addr_prev_count_24h'
]

summary = historical_features[historical_cols].describe().T

display(summary.round(2))


print("\nPercentage of transactions with at least one")
print("previous related transaction within 24 hours:\n")

for col in historical_cols:

    percentage = (
        historical_features[col].gt(0).mean() * 100
    )

    print(f"{col}: {percentage:.2f}%")

# 10. Enhanced XGBoost with Historical Network-Inspired Features

To evaluate whether transaction relationships improve fraud detection, the five historical 24-hour features are added to the original baseline feature set.

The enhanced model uses the same chronological train-validation split, preprocessing procedure, XGBoost hyperparameters, and class weighting as the baseline model.

Therefore, the main difference between the two models is the inclusion of historical relational features.

In [ ]:
# ============================================================
# 10.1 SPLIT HISTORICAL FEATURES
# ============================================================

hist_train = (
    historical_features
    .iloc[:split_index]
    .reset_index(drop=True)
)

hist_valid = (
    historical_features
    .iloc[split_index:]
    .reset_index(drop=True)
)

print("Historical Training:", hist_train.shape)
print("Historical Validation:", hist_valid.shape)

# Verify alignment with original chronological split
print(
    "\nTraining IDs aligned:",
    np.array_equal(
        hist_train['TransactionID'].to_numpy(),
        train_df['TransactionID'].to_numpy()
    )
)

print(
    "Validation IDs aligned:",
    np.array_equal(
        hist_valid['TransactionID'].to_numpy(),
        valid_df['TransactionID'].to_numpy()
    )
)

In [ ]:
# ============================================================
# 10.2 CREATE ENHANCED FEATURE SET
# ============================================================

new_features = [
    'card1_prev_count_24h',
    'addr1_prev_count_24h',
    'email_prev_count_24h',
    'device_prev_count_24h',
    'card_addr_prev_count_24h'
]

# Keep the baseline matrices unchanged
X_train_enhanced = X_train.copy()
X_valid_enhanced = X_valid.copy()

# Reset indices to guarantee row alignment
X_train_enhanced = X_train_enhanced.reset_index(drop=True)
X_valid_enhanced = X_valid_enhanced.reset_index(drop=True)

# Add historical features
for feature in new_features:

    X_train_enhanced[feature] = (
        hist_train[feature].to_numpy()
    )

    X_valid_enhanced[feature] = (
        hist_valid[feature].to_numpy()
    )


print("Baseline Training Features:", X_train.shape[1])
print("Enhanced Training Features:", X_train_enhanced.shape[1])

print("\nEnhanced Training Shape:", X_train_enhanced.shape)
print("Enhanced Validation Shape:", X_valid_enhanced.shape)

In [ ]:
# ============================================================
# 10.3 TRAIN ENHANCED XGBOOST
# ============================================================

enhanced_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

enhanced_model.fit(
    X_train_enhanced,
    y_train
)

print("Enhanced XGBoost training completed.")

In [ ]:
# ============================================================
# 10.4 EVALUATE ENHANCED MODEL
# ============================================================

enhanced_prob = enhanced_model.predict_proba(
    X_valid_enhanced
)[:, 1]

enhanced_ap = average_precision_score(
    y_valid,
    enhanced_prob
)

enhanced_roc = roc_auc_score(
    y_valid,
    enhanced_prob
)

print("Enhanced XGBoost Performance")
print("-" * 40)

print(f"Average Precision (AP): {enhanced_ap:.4f}")
print(f"ROC-AUC:                {enhanced_roc:.4f}")

In [ ]:
# ============================================================
# 10.5 MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({
    'Model': [
        'Baseline XGBoost',
        'XGBoost + Historical Features'
    ],
    'Average Precision': [
        baseline_ap,
        enhanced_ap
    ],
    'ROC-AUC': [
        baseline_roc,
        enhanced_roc
    ]
})

comparison['AP Change'] = [
    0,
    enhanced_ap - baseline_ap
]

comparison['ROC-AUC Change'] = [
    0,
    enhanced_roc - baseline_roc
]

display(comparison.round(4))

## 10.6 Importance of Historical Network-Inspired Features

Although the enhanced model produced a small improvement in predictive performance, it is useful to determine whether the newly engineered historical features contributed to the model.

XGBoost feature importance is examined for the five historical relationship features before conducting a more detailed SHAP analysis.

In [ ]:
# ============================================================
# 10.6 HISTORICAL FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    'Feature': X_train_enhanced.columns,
    'Importance': enhanced_model.feature_importances_
})

historical_importance = (
    feature_importance[
        feature_importance['Feature'].isin(new_features)
    ]
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

display(historical_importance.round(6))


# Rank each historical feature among ALL model features
all_importance = (
    feature_importance
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

all_importance['Rank'] = (
    np.arange(1, len(all_importance) + 1)
)

historical_ranks = all_importance[
    all_importance['Feature'].isin(new_features)
][['Rank', 'Feature', 'Importance']]

print("\nHistorical Feature Ranks Among All Features:")
display(historical_ranks)

# 11. SHAP Model Explainability

SHAP (SHapley Additive exPlanations) is used to interpret the enhanced XGBoost model.

SHAP values quantify how individual features contribute to model predictions. This analysis is used to identify the most influential features globally and to examine the contribution of the newly engineered historical relationship features.

To reduce computational requirements, SHAP values are calculated using a reproducible sample of 5,000 validation transactions.

In [ ]:
# ============================================================
# 11.1 CALCULATE SHAP VALUES
# ============================================================

import shap

# Reproducible validation sample
SHAP_SAMPLE_SIZE = 5000

X_shap = X_valid_enhanced.sample(
    n=min(SHAP_SAMPLE_SIZE, len(X_valid_enhanced)),
    random_state=42
)

print("SHAP sample shape:", X_shap.shape)

# TreeExplainer is efficient for XGBoost
explainer = shap.TreeExplainer(enhanced_model)

shap_values = explainer(X_shap)

print("SHAP values calculated successfully.")
print("SHAP values shape:", shap_values.values.shape)

## 11.2 Global SHAP Feature Importance

Global SHAP importance is used to identify the features that have the greatest overall influence on the enhanced XGBoost model.

The importance of each feature is calculated using the mean absolute SHAP value across the sampled validation transactions. Larger values indicate a greater overall contribution to model predictions.

In [ ]:
# ============================================================
# 11.2 GLOBAL SHAP FEATURE IMPORTANCE
# ============================================================

shap_importance = pd.DataFrame({
    'Feature': X_shap.columns,
    'Mean_Absolute_SHAP': np.abs(
        shap_values.values
    ).mean(axis=0)
})

shap_importance = (
    shap_importance
    .sort_values(
        'Mean_Absolute_SHAP',
        ascending=False
    )
    .reset_index(drop=True)
)

shap_importance['Rank'] = (
    np.arange(1, len(shap_importance) + 1)
)

print("Top 20 Features by Mean Absolute SHAP Value:")
display(
    shap_importance[
        ['Rank', 'Feature', 'Mean_Absolute_SHAP']
    ].head(20)
)

In [ ]:
# ============================================================
# 11.3 SHAP SUMMARY PLOT
# ============================================================

shap.summary_plot(
    shap_values.values,
    X_shap,
    max_display=20,
    show=False
)

plt.title(
    "Top 20 Features Influencing Fraud Predictions"
)

plt.tight_layout()
plt.show()

## 11.4 SHAP Analysis of Historical Network-Inspired Features

The SHAP importance of the five engineered historical features is examined separately to evaluate how strongly the relational information contributes to fraud predictions.

This analysis complements the overall model comparison by showing whether the historical transaction relationships are actively used by the enhanced model.

In [ ]:
# ============================================================
# 11.4 SHAP IMPORTANCE OF HISTORICAL FEATURES
# ============================================================

historical_shap = (
    shap_importance[
        shap_importance['Feature'].isin(new_features)
    ][
        ['Rank', 'Feature', 'Mean_Absolute_SHAP']
    ]
    .sort_values('Rank')
    .reset_index(drop=True)
)

print("Historical Features — SHAP Importance")
print("-" * 50)

display(historical_shap.round(6))

In [ ]:
# ============================================================
# 11.5 HISTORICAL FEATURES SHAP SUMMARY
# ============================================================

historical_indices = [
    X_shap.columns.get_loc(feature)
    for feature in new_features
]

shap.summary_plot(
    shap_values.values[:, historical_indices],
    X_shap[new_features],
    max_display=5,
    show=False
)

plt.title(
    "Impact of Historical Relationship Features"
)

plt.tight_layout()
plt.show()

### Interpretation

The SHAP analysis confirms that the engineered historical relationship features contribute to the enhanced fraud detection model. Among these features, `card1_prev_count_24h` had the strongest influence, ranking 17th among all 428 model features. The remaining historical features also appeared within the top 100 features based on mean absolute SHAP values.

The SHAP distributions indicate nonlinear effects rather than a simple relationship in which more repeated activity always implies greater fraud risk. Historical transaction behavior can increase or decrease the predicted fraud risk depending on its value and interactions with other transaction characteristics. These findings support the use of temporal relationship features as complementary information alongside traditional transaction-level features.

# 12. Deep Learning Model — Artificial Neural Network

To complement the tree-based XGBoost approach, an Artificial Neural Network (ANN) is developed for fraud classification.

Because neural networks are sensitive to differences in feature scale, the input features are standardized before training. The scaler is fitted using the training period only and then applied to the validation period to prevent data leakage.

The ANN uses fully connected dense layers with batch normalization and dropout for regularization. A sigmoid output layer produces the probability that a transaction is fraudulent.

Class weights are used to address the substantial class imbalance in the dataset. Model performance is evaluated using Average Precision and ROC-AUC, consistent with the XGBoost experiments.

In [ ]:
# ============================================================
# 12.1 IMPORT DEEP LEARNING LIBRARIES
# ============================================================

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler

print("TensorFlow version:", tf.__version__)

In [ ]:
# ============================================================
# 12.2 PREPARE BASELINE ANN DATA
# ============================================================

# Copies — DO NOT modify our XGBoost data
X_train_ann = X_train.copy()
X_valid_ann = X_valid.copy()

# Calculate medians from TRAINING DATA ONLY
train_medians = X_train_ann.median()

# Fill missing values
X_train_ann = X_train_ann.fillna(train_medians)
X_valid_ann = X_valid_ann.fillna(train_medians)

# If a column's training median is itself NaN,
# fill any remaining missing values with 0
X_train_ann = X_train_ann.fillna(0)
X_valid_ann = X_valid_ann.fillna(0)

print("Missing values after ANN imputation:")
print("Training:", X_train_ann.isnull().sum().sum())
print("Validation:", X_valid_ann.isnull().sum().sum())

In [ ]:
# ============================================================
# 12.3 STANDARDIZE ANN FEATURES
# ============================================================

ann_scaler = StandardScaler()

# Fit ONLY on training data
X_train_ann_scaled = ann_scaler.fit_transform(
    X_train_ann
)

# Apply same transformation to validation
X_valid_ann_scaled = ann_scaler.transform(
    X_valid_ann
)

# Convert to float32 to reduce memory usage
X_train_ann_scaled = X_train_ann_scaled.astype('float32')
X_valid_ann_scaled = X_valid_ann_scaled.astype('float32')

print("ANN Training Shape:", X_train_ann_scaled.shape)
print("ANN Validation Shape:", X_valid_ann_scaled.shape)

print("\nData type:", X_train_ann_scaled.dtype)

print(
    "Any NaN in training:",
    np.isnan(X_train_ann_scaled).any()
)

print(
    "Any NaN in validation:",
    np.isnan(X_valid_ann_scaled).any()
)

## 12.4 Baseline ANN Architecture

A feed-forward Artificial Neural Network is used as the deep learning model for the tabular fraud detection problem.

The network contains multiple dense layers with batch normalization and dropout. Batch normalization helps stabilize training, while dropout reduces overfitting. The final sigmoid neuron outputs the predicted probability of fraud.

Early stopping is used to stop training when validation performance no longer improves.

In [ ]:
# ============================================================
# 12.4 BUILD BASELINE ANN
# ============================================================

# Reproducibility
tf.keras.utils.set_random_seed(42)

baseline_ann = keras.Sequential([
    
    layers.Input(shape=(X_train_ann_scaled.shape[1],)),
    
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.30),
    
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.30),
    
    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.20),
    
    layers.Dense(1, activation='sigmoid')
])

baseline_ann.summary()

In [ ]:
# ============================================================
# 12.5 COMPILE BASELINE ANN
# ============================================================

baseline_ann.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(
            curve='ROC',
            name='roc_auc'
        ),
        keras.metrics.AUC(
            curve='PR',
            name='pr_auc'
        )
    ]
)

print("Baseline ANN compiled successfully.")

In [ ]:
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ============================================================
# 12.6 CLASS WEIGHTS AND CALLBACKS
# ============================================================

# Calculate class imbalance
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

ann_scale_pos_weight = negative / positive

ann_class_weights = {
    0: 1.0,
    1: ann_scale_pos_weight
}

# Early stopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate if validation PR-AUC stops improving
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_pr_auc',
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Non-fraud transactions:", negative)
print("Fraud transactions:", positive)
print("Fraud class weight:", round(ann_scale_pos_weight, 2))
print("ANN class weights:", ann_class_weights)

In [ ]:
# ============================================================
# 12.7 TRAIN BASELINE ANN
# ============================================================

baseline_ann_history = baseline_ann.fit(
    X_train_ann_scaled,
    y_train.to_numpy(),

    validation_data=(
        X_valid_ann_scaled,
        y_valid.to_numpy()
    ),

    epochs=25,
    batch_size=2048,

    class_weight=ann_class_weights,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

## 12.8 Baseline ANN Evaluation

The trained ANN is evaluated on the chronological validation set using Average Precision (AP) and ROC-AUC. These are calculated using the same scikit-learn metrics used for the XGBoost model, allowing a consistent comparison between the machine learning and deep learning approaches.

In [ ]:
# ============================================================
# 12.8 EVALUATE BASELINE ANN
# ============================================================

from sklearn.metrics import average_precision_score, roc_auc_score

# Predict fraud probabilities
ann_valid_pred = baseline_ann.predict(
    X_valid_ann_scaled,
    batch_size=2048,
    verbose=0
).ravel()

# Official evaluation metrics
baseline_ann_ap = average_precision_score(
    y_valid,
    ann_valid_pred
)

baseline_ann_roc = roc_auc_score(
    y_valid,
    ann_valid_pred
)

print("Baseline ANN Results")
print("----------------------------")
print(f"Average Precision (AP): {baseline_ann_ap:.4f}")
print(f"ROC-AUC:               {baseline_ann_roc:.4f}")

print("\nBaseline XGBoost Results")
print("----------------------------")
print(f"Average Precision (AP): {baseline_ap:.4f}")
print(f"ROC-AUC:               {baseline_roc:.4f}")

# 13. Enhanced ANN with Historical Network-Inspired Features

The baseline ANN uses only the original transaction-level features. To investigate whether temporal relationship information can improve deep learning performance, a second ANN is trained using the five historical 24-hour features developed from the network analysis.

The same ANN architecture and training strategy are maintained to allow a fair comparison. Therefore, the primary difference between the baseline and enhanced ANN models is the inclusion of the historical relationship features.

In [ ]:
# ============================================================
# 13.1 PREPARE ENHANCED ANN DATA
# ============================================================

X_train_ann_enhanced = X_train_enhanced.copy()
X_valid_ann_enhanced = X_valid_enhanced.copy()

# Training medians only
enhanced_train_medians = X_train_ann_enhanced.median()

# Impute missing values
X_train_ann_enhanced = X_train_ann_enhanced.fillna(
    enhanced_train_medians
)

X_valid_ann_enhanced = X_valid_ann_enhanced.fillna(
    enhanced_train_medians
)

# Handle columns whose training median may still be NaN
X_train_ann_enhanced = X_train_ann_enhanced.fillna(0)
X_valid_ann_enhanced = X_valid_ann_enhanced.fillna(0)

print("Enhanced ANN shapes:")
print("Training:", X_train_ann_enhanced.shape)
print("Validation:", X_valid_ann_enhanced.shape)

print("\nMissing values:")
print("Training:", X_train_ann_enhanced.isnull().sum().sum())
print("Validation:", X_valid_ann_enhanced.isnull().sum().sum())

In [ ]:
# ============================================================
# 13.2 STANDARDIZE ENHANCED ANN FEATURES
# ============================================================

enhanced_ann_scaler = StandardScaler()

X_train_ann_enhanced_scaled = enhanced_ann_scaler.fit_transform(
    X_train_ann_enhanced
)

X_valid_ann_enhanced_scaled = enhanced_ann_scaler.transform(
    X_valid_ann_enhanced
)

# Reduce memory
X_train_ann_enhanced_scaled = (
    X_train_ann_enhanced_scaled.astype('float32')
)

X_valid_ann_enhanced_scaled = (
    X_valid_ann_enhanced_scaled.astype('float32')
)

print("Enhanced ANN Training Shape:",
      X_train_ann_enhanced_scaled.shape)

print("Enhanced ANN Validation Shape:",
      X_valid_ann_enhanced_scaled.shape)

print("\nAny NaN in training:",
      np.isnan(X_train_ann_enhanced_scaled).any())

print("Any NaN in validation:",
      np.isnan(X_valid_ann_enhanced_scaled).any())

In [ ]:
# ============================================================
# 13.3 BUILD ENHANCED ANN
# ============================================================

tf.keras.utils.set_random_seed(42)

enhanced_ann = keras.Sequential([
    
    layers.Input(shape=(X_train_ann_enhanced_scaled.shape[1],)),
    
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.30),
    
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.30),
    
    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.20),
    
    layers.Dense(1, activation='sigmoid')
])

enhanced_ann.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(curve='ROC', name='roc_auc'),
        keras.metrics.AUC(curve='PR', name='pr_auc')
    ]
)

enhanced_ann.summary()

In [ ]:
# ============================================================
# 13.4 CALLBACKS FOR ENHANCED ANN
# ============================================================

enhanced_early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

enhanced_reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_pr_auc',
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# ============================================================
# 13.5 TRAIN ENHANCED ANN
# ============================================================

enhanced_ann_history = enhanced_ann.fit(
    X_train_ann_enhanced_scaled,
    y_train.to_numpy(),

    validation_data=(
        X_valid_ann_enhanced_scaled,
        y_valid.to_numpy()
    ),

    epochs=25,
    batch_size=2048,

    class_weight=ann_class_weights,

    callbacks=[
        enhanced_early_stopping,
        enhanced_reduce_lr
    ],

    verbose=1
)

In [ ]:
# ============================================================
# 13.6 EVALUATE ENHANCED ANN
# ============================================================

enhanced_ann_pred = enhanced_ann.predict(
    X_valid_ann_enhanced_scaled,
    batch_size=2048,
    verbose=0
).ravel()

enhanced_ann_ap = average_precision_score(
    y_valid,
    enhanced_ann_pred
)

enhanced_ann_roc = roc_auc_score(
    y_valid,
    enhanced_ann_pred
)

print("Enhanced ANN Results")
print("----------------------------")
print(f"Average Precision (AP): {enhanced_ann_ap:.4f}")
print(f"ROC-AUC:               {enhanced_ann_roc:.4f}")

print("\nBaseline ANN Results")
print("----------------------------")
print(f"Average Precision (AP): {baseline_ann_ap:.4f}")
print(f"ROC-AUC:               {baseline_ann_roc:.4f}")

print("\nChange After Adding Historical Features")
print("----------------------------------------")
print(f"AP Change:      {enhanced_ann_ap - baseline_ann_ap:+.4f}")
print(f"ROC-AUC Change: {enhanced_ann_roc - baseline_ann_roc:+.4f}")

### Interpretation

The baseline ANN achieved an Average Precision (AP) of 0.4509 and a ROC-AUC of 0.8612. After adding the five historical network-inspired features, the enhanced ANN achieved an AP of 0.4404 and a ROC-AUC of 0.8725.

The historical features therefore had a mixed effect on the deep learning model. ROC-AUC increased by 0.0113, indicating improved overall ranking ability, while Average Precision decreased by 0.0105. Because fraud represents only a small proportion of the transactions, Average Precision is particularly informative for evaluating performance on the minority fraud class.

In comparison, XGBoost achieved higher AP and ROC-AUC values than both ANN models. The historical features also produced a small improvement in both metrics for XGBoost. These results demonstrate that the value of the engineered temporal relationship features depends on the modeling approach and evaluation metric.

# 14. Final Fraud Detection Evaluation

The Enhanced XGBoost model achieved the strongest overall performance among the implemented models and was therefore selected for the final fraud detection evaluation.

Because fraud detection is a highly imbalanced classification problem, ranking metrics alone do not fully describe the model's operational behavior. Average Precision (AP) and ROC-AUC evaluate how well the model ranks fraudulent transactions, while a classification threshold is required to convert predicted probabilities into fraud/non-fraud decisions.

The model is first evaluated at a threshold of **0.50** using precision, recall, F1-score, and the confusion matrix. A threshold sensitivity analysis is then performed to illustrate the trade-off between detecting more fraudulent transactions and reducing false alerts.

## 14.1 Evaluation at Threshold = 0.50

A probability threshold of **0.50** is used as the reference operating point for the final model evaluation.

At this threshold, the confusion matrix shows how many legitimate and fraudulent transactions are correctly or incorrectly classified. Precision measures how many flagged transactions are actually fraudulent, while recall measures how many of the known fraudulent transactions are successfully detected. The F1-score summarizes the balance between these two measures.

In [ ]:
# ============================================================
# 14.1 FRAUD DETECTION AT THRESHOLD = 0.50
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Use probabilities from our enhanced XGBoost
enhanced_xgb_prob = enhanced_model.predict_proba(
    X_valid_enhanced
)[:, 1]

threshold = 0.50

# Convert probabilities into fraud / non-fraud predictions
enhanced_xgb_pred = (
    enhanced_xgb_prob >= threshold
).astype(int)

# Confusion matrix
cm = confusion_matrix(
    y_valid,
    enhanced_xgb_pred
)

tn, fp, fn, tp = cm.ravel()

precision = precision_score(
    y_valid,
    enhanced_xgb_pred
)

recall = recall_score(
    y_valid,
    enhanced_xgb_pred
)

f1 = f1_score(
    y_valid,
    enhanced_xgb_pred
)

print("Enhanced XGBoost — Threshold = 0.50")
print("====================================")

print("\nConfusion Matrix:")
print(cm)

print("\nTrue Negatives: ", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives: ", tp)

print("\nFraud Detection Metrics")
print("-----------------------")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_valid,
        enhanced_xgb_pred,
        digits=4
    )
)

### Interpretation

At the 0.50 threshold, the model identifies **3,014 of the 4,064 fraudulent transactions**, corresponding to a recall of approximately **74.2%**. This indicates that the model captures a substantial proportion of the known fraud cases.

However, the precision is approximately **21.3%**, meaning that many transactions flagged as fraud are false positives. This reflects an important operational trade-off in fraud detection: increasing fraud coverage can also increase the number of legitimate transactions requiring investigation.

Therefore, the 0.50 threshold is treated as a reference decision threshold rather than an automatically optimal operating point.

 ## 14.2 Threshold Sensitivity Analysis

The classification threshold determines the balance between fraud detection and false alerts.

To examine this trade-off, the Enhanced XGBoost model is evaluated across thresholds from **0.30 to 0.80**. Lower thresholds are expected to identify more fraudulent transactions but generate more false positives, while higher thresholds are expected to improve precision at the cost of missing more fraud cases.

This analysis is used to understand model behavior under different decision policies rather than to claim a universally optimal threshold.

In [ ]:
 # ============================================================
# 14.2 COMPARE FRAUD DETECTION THRESHOLDS
# ============================================================

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

threshold_results = []

for threshold in thresholds:

    predictions = (
        enhanced_xgb_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        predictions
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_valid, predictions),
        "Recall": recall_score(y_valid, predictions),
        "F1": f1_score(y_valid, predictions),
        "Fraud Detected (TP)": tp,
        "Fraud Missed (FN)": fn,
        "False Alerts (FP)": fp
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(4)

### Threshold Trade-off Interpretation

The results show a clear precision-recall trade-off.

At a lower threshold of **0.30**, the model achieves high recall (**87.3%**) and detects 3,546 fraudulent transactions, but it also generates 25,511 false alerts.

As the threshold increases, precision improves while recall decreases. At **0.80**, precision increases to approximately **51.9%**, while recall decreases to approximately **47.7%**, with 1,796 false alerts.

This demonstrates that threshold selection should depend on the operational objective and the relative cost of missed fraud versus false alerts. In this project, the **0.50 threshold is retained as the reference threshold** for the dashboard and final demonstration rather than being presented as an optimized production threshold.

# 15. Fraud Detection Investigation Dashboard

To translate the final fraud detection model into an interactive investigation tool, a Streamlit dashboard is developed for transaction-level analysis.

The dashboard combines three components of the project:

- **Fraud risk prediction:** displays the Enhanced XGBoost risk score and classification decision.
- **Model explainability:** uses SHAP values to explain the features contributing to an individual prediction.
- **Relational investigation:** identifies historical transactions connected through shared entities within the previous 24-hour window.

The dashboard is designed as an investigation and demonstration tool rather than a production fraud detection system. Connections between transactions provide contextual evidence and do not independently establish coordinated fraud.

## 15.1 Dashboard Setup

Streamlit is used to build the interactive investigation interface, while PyVis is used to visualize transaction relationships as an interactive network.

The required dashboard libraries are installed before preparing the data and application assets.

In [ ]:
#============================================================
# 15.1 Dashboard Setup
#============================================================
!pip install streamlit pyvis -q

print("Streamlit and PyVis installed successfully.")

## 15.2 Dashboard Data Preparation

The dashboard uses the chronological validation set to demonstrate transaction-level model predictions.

For each validation transaction, the prepared dashboard data includes:

- the original transaction identifier;
- the known fraud label;
- the Enhanced XGBoost fraud probability;
- the binary fraud prediction using the reference threshold of **0.50**.

The actual fraud label is included only for evaluation and demonstration purposes. In a real deployment scenario, this label would not be available when investigating a new transaction.

In [ ]:
# ============================================================
# 15.2 DASHBOARD DATA PREPARATION — CORRECTED ALIGNMENT
# ============================================================

# Reset the chronological validation data so that its row
# positions align exactly with X_valid_enhanced
valid_rows = valid_df.reset_index(drop=True)

# Start from the enhanced validation feature set
dashboard_data = X_valid_enhanced.copy()

# Add the CORRECT original validation transaction identifiers
dashboard_data["TransactionID"] = (
    valid_rows["TransactionID"].values
)

# Add actual labels and model outputs
dashboard_data["Actual_Fraud"] = y_valid.values
dashboard_data["Risk_Score"] = enhanced_xgb_prob

# Apply the reference classification threshold
dashboard_data["Fraud_Prediction"] = (
    enhanced_xgb_prob >= 0.50
).astype(int)

# Transaction IDs available in the full validation dashboard
dashboard_transaction_ids = (
    dashboard_data["TransactionID"]
    .astype(int)
    .tolist()
)


# ============================================================
# ALIGNMENT VALIDATION
# ============================================================

assert len(valid_rows) == len(X_valid_enhanced)
assert len(dashboard_data) == len(y_valid)
assert len(dashboard_data) == len(enhanced_xgb_prob)

assert np.array_equal(
    dashboard_data["TransactionID"].values,
    valid_rows["TransactionID"].values
)

assert dashboard_data["TransactionID"].isin(
    valid_df["TransactionID"]
).all()


# ============================================================
# RESULTS
# ============================================================

print("Dashboard data prepared successfully.")
print("Validation transactions:", len(dashboard_data))
print("Dashboard columns:", dashboard_data.shape[1])
print("Available Transaction IDs:", len(dashboard_transaction_ids))

print("\nAlignment checks passed ✓")

print("\nFirst 10 dashboard Transaction IDs:")
print(dashboard_transaction_ids[:10])

print(
    "\nFirst validation Transaction ID:",
    int(valid_rows["TransactionID"].iloc[0])
)

print(
    "First dashboard Transaction ID:",
    int(dashboard_data["TransactionID"].iloc[0])
)

## 15.3 Dashboard Investigation Sample

The full validation set contains more than 118,000 transactions. However, transaction-level SHAP values were previously computed for a representative sample of **5,000 validation transactions** to keep explainability analysis computationally manageable.

The interactive investigation dashboard therefore uses these 5,000 transactions as its demonstration sample. This ensures that every transaction available in the dashboard has a corresponding local SHAP explanation while preserving the original Enhanced XGBoost predictions.

The sample is used only for dashboard visualization and does not change the model training or final validation results.

In [ ]:
# ============================================================
# 15.3 DASHBOARD INVESTIGATION SAMPLE
# ============================================================

# Restrict the interactive dashboard to transactions
# included in the SHAP explanation sample
dashboard_sample = dashboard_data.loc[
    X_shap.index
].copy()

# Verify alignment between dashboard transactions and SHAP rows
assert dashboard_sample.index.equals(X_shap.index)

print("Dashboard investigation sample prepared successfully.")
print("Validation transactions:", len(dashboard_data))
print("Dashboard sample transactions:", len(dashboard_sample))
print("SHAP transactions:", len(X_shap))
print(
    "SHAP coverage of dashboard sample:",
    f"{len(X_shap) / len(dashboard_sample) * 100:.1f}%"
)

## 15.4 Transaction Investigation Data

To make the dashboard useful for transaction-level investigation, the SHAP sample is linked back to selected fields from the original transaction data.

For each of the 5,000 dashboard transactions, the investigation dataset includes the transaction amount, card identifier, address, email domain, and device information. The model risk score, binary prediction, and known validation label are then added from the prepared dashboard sample.

These fields provide interpretable transaction context alongside the model output and will later be combined with SHAP explanations and relational evidence.

In [ ]:
# ============================================================
# 15.4 TRANSACTION INVESTIGATION DATA — CORRECTED ALIGNMENT
# ============================================================

# Retrieve interpretable fields from the CORRECT validation rows
# for the transactions included in the SHAP sample
investigation_data = valid_rows.loc[
    X_shap.index,
    [
        "TransactionID",
        "TransactionAmt",
        "card1",
        "addr1",
        "P_emaildomain",
        "DeviceInfo"
    ]
].copy()


# Add model evaluation results from the same aligned rows
investigation_data["Actual_Fraud"] = dashboard_sample[
    "Actual_Fraud"
].values

investigation_data["Risk_Score"] = dashboard_sample[
    "Risk_Score"
].values

investigation_data["Fraud_Prediction"] = dashboard_sample[
    "Fraud_Prediction"
].values


# ============================================================
# ALIGNMENT VALIDATION
# ============================================================

assert len(investigation_data) == len(X_shap)

assert (
    investigation_data["TransactionID"]
    .astype(int)
    .values
    ==
    dashboard_sample["TransactionID"]
    .astype(int)
    .values
).all()

assert investigation_data["TransactionID"].isin(
    valid_df["TransactionID"]
).all()


# ============================================================
# RESULTS
# ============================================================

print("Transaction investigation data prepared successfully.")
print("Transactions:", len(investigation_data))
print("Columns:", investigation_data.shape[1])

print("\nAlignment checks passed ✓")
print("All investigation transactions belong to validation data ✓")

print("\nFirst investigation transaction:")
display(investigation_data.head(1))

print("\nFirst 10 investigation Transaction IDs:")
print(
    investigation_data["TransactionID"]
    .head(10)
    .astype(int)
    .tolist()
)

## 15.5 Compact SHAP Explanations for the Dashboard

SHAP values are used to provide local explanations for individual fraud predictions.

To keep the dashboard assets compact and efficient, only the **10 features with the largest absolute SHAP contributions** are retained for each of the 5,000 dashboard transactions. The sign of each SHAP value indicates the direction of its contribution: positive values push the model output toward fraud, while negative values push it toward non-fraud.

This compression is used only for dashboard visualization. The complete SHAP analysis performed earlier in the notebook remains unchanged.

In [ ]:
# ============================================================
# 15.5 COMPACT SHAP EXPLANATIONS FOR THE DASHBOARD
# ============================================================

TOP_N_SHAP = 10

shap_dashboard_records = []

for i, transaction_index in enumerate(X_shap.index):

    transaction_id = int(
        investigation_data.loc[
            transaction_index,
            "TransactionID"
        ]
    )

    shap_row = shap_values.values[i]

    # Identify the features with the strongest local impact
    top_indices = np.argsort(
        np.abs(shap_row)
    )[-TOP_N_SHAP:][::-1]

    for feature_index in top_indices:

        shap_dashboard_records.append({
            "TransactionID": transaction_id,
            "Feature": X_shap.columns[feature_index],
            "Feature_Value": X_shap.iloc[i, feature_index],
            "SHAP_Value": shap_row[feature_index]
        })


shap_dashboard_data = pd.DataFrame(
    shap_dashboard_records
)

# Validation checks
expected_rows = len(X_shap) * TOP_N_SHAP

assert len(shap_dashboard_data) == expected_rows
assert (
    shap_dashboard_data["TransactionID"].nunique()
    == len(X_shap)
)

print("Compact SHAP explanations prepared successfully.")
print("Transactions:", shap_dashboard_data["TransactionID"].nunique())
print("Features retained per transaction:", TOP_N_SHAP)
print("Total SHAP rows:", len(shap_dashboard_data))

display(shap_dashboard_data.head(10))

## 15.6 Historical Relational Analysis for Dashboard Transactions

To support interactive relational investigation across the dashboard sample, historical connections are generated for all 5,000 dashboard transactions rather than for a single representative case.

For each dashboard transaction, only transactions occurring within the **previous 24 hours** are considered. A historical transaction is treated as connected when it shares at least one investigated entity with the selected transaction: **card identifier, address, email domain, or device information**.

The analysis is strictly historical: future transactions are excluded. Fraud labels are attached only after the relationships are identified and are used for validation and visualization, not for determining whether a connection exists.

The resulting connection table allows the dashboard to dynamically display the historical relational neighborhood associated with each investigated transaction.

In [ ]:
# ============================================================
# 15.6 HISTORICAL RELATIONAL ANALYSIS
# FOR ALL DASHBOARD TRANSACTIONS
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TIME_WINDOW = 24 * 60 * 60

entity_columns = {
    "card1": "card1",
    "addr1": "addr1",
    "email": "P_emaildomain",
    "device": "DeviceInfo"
}


# ------------------------------------------------------------
# PREPARE SOURCE DATA
# ------------------------------------------------------------

history_source = train[
    [
        "TransactionID",
        "TransactionDT",
        "card1",
        "addr1",
        "P_emaildomain",
        "DeviceInfo",
        "isFraud"
    ]
].copy()

history_source = history_source.sort_values(
    "TransactionDT"
).reset_index(drop=True)


# Dashboard transactions
dashboard_ids = set(
    investigation_data[
        "TransactionID"
    ].astype(int)
)


# ------------------------------------------------------------
# BUILD HISTORICAL ENTITY INDEX
# ------------------------------------------------------------
# For each entity value, store earlier transactions.
# The index is built while moving forward through time,
# which prevents future transactions from being used.

entity_history = {
    entity_name: defaultdict(list)
    for entity_name in entity_columns
}

connection_records = []

dashboard_processed = 0


for row in history_source.itertuples(index=False):

    transaction_id = int(row.TransactionID)
    transaction_time = row.TransactionDT

    current_entities = {
        "card1": row.card1,
        "addr1": row.addr1,
        "email": row.P_emaildomain,
        "device": row.DeviceInfo
    }


    # --------------------------------------------------------
    # GENERATE CONNECTIONS ONLY FOR DASHBOARD TRANSACTIONS
    # --------------------------------------------------------

    if transaction_id in dashboard_ids:

        related_map = {}


        for entity_name, entity_value in current_entities.items():

            if pd.isna(entity_value):
                continue


            historical_matches = entity_history[
                entity_name
            ].get(
                entity_value,
                []
            )


            for (
                previous_time,
                previous_id,
                previous_fraud
            ) in reversed(historical_matches):

                time_difference = (
                    transaction_time
                    - previous_time
                )


                # Older than the 24-hour window
                if time_difference > TIME_WINDOW:
                    break


                # Strictly historical only
                if previous_time >= transaction_time:
                    continue


                if previous_id not in related_map:

                    related_map[
                        previous_id
                    ] = {
                        "types": [],
                        "fraud": previous_fraud
                    }


                related_map[
                    previous_id
                ]["types"].append(
                    entity_name
                )


        # ----------------------------------------------------
        # SAVE CONNECTION RECORDS
        # ----------------------------------------------------

        for (
            related_id,
            related_info
        ) in related_map.items():

            connection_records.append({
                "Source_Transaction":
                    transaction_id,

                "Related_Transaction":
                    int(related_id),

                "Connection_Type":
                    ", ".join(
                        related_info["types"]
                    ),

                "Actual_Fraud":
                    int(
                        related_info["fraud"]
                    )
            })


        dashboard_processed += 1

        if dashboard_processed % 500 == 0:

            print(
                f"Processed "
                f"{dashboard_processed:,} / "
                f"{len(dashboard_ids):,} "
                f"dashboard transactions"
            )


    # --------------------------------------------------------
    # ADD CURRENT TRANSACTION TO HISTORY
    # AFTER CONNECTION SEARCH
    # --------------------------------------------------------
    # This guarantees that only earlier transactions
    # can become historical neighbors.

    for entity_name, entity_value in current_entities.items():

        if pd.isna(entity_value):
            continue

        entity_history[
            entity_name
        ][
            entity_value
        ].append(
            (
                transaction_time,
                transaction_id,
                int(row.isFraud)
            )
        )


# ------------------------------------------------------------
# CREATE FINAL CONNECTION DATAFRAME
# ------------------------------------------------------------

graph_connections_dashboard = pd.DataFrame(
    connection_records,
    columns=[
        "Source_Transaction",
        "Related_Transaction",
        "Connection_Type",
        "Actual_Fraud"
    ]
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

connected_source_count = (
    graph_connections_dashboard[
        "Source_Transaction"
    ].nunique()
    if len(graph_connections_dashboard) > 0
    else 0
)

total_connections = len(
    graph_connections_dashboard
)

fraud_connections = (
    int(
        graph_connections_dashboard[
            "Actual_Fraud"
        ].sum()
    )
    if total_connections > 0
    else 0
)


print("\n" + "=" * 60)
print("HISTORICAL RELATIONAL ANALYSIS COMPLETE")
print("=" * 60)

print(
    "Dashboard transactions processed:",
    f"{dashboard_processed:,}"
)

print(
    "Dashboard transactions with connections:",
    f"{connected_source_count:,}"
)

print(
    "Total historical connections:",
    f"{total_connections:,}"
)

print(
    "Connected transactions with known fraud labels:",
    f"{fraud_connections:,}"
)

print(
    "Transactions without historical connections:",
    f"{len(dashboard_ids) - connected_source_count:,}"
)


if total_connections > 0:

    print("\nConnection type combinations:")

    display(
        graph_connections_dashboard[
            "Connection_Type"
        ]
        .value_counts()
        .head(15)
        .to_frame("Count")
    )


display(
    graph_connections_dashboard.head()
)

## 15.7 Dashboard Asset Export

The final MAKĀMIN dashboard combines two analytical levels.

First, summary model outputs for the complete chronological validation set are exported to support an overall fraud analysis view. This includes transaction identifiers, known validation labels, Enhanced XGBoost risk scores, and reference-threshold predictions.

Second, detailed investigation assets are exported for the 5,000 validation transactions included in the SHAP sample. These assets include interpretable transaction information, compact local SHAP explanations, and historical relational connections from the previous 24 hours.

This design allows MAKĀMIN to provide both a population-level fraud overview and transaction-level investigation without requiring the full model-development dataset inside the Streamlit application.

In [ ]:
# ============================================================
# 15.7 DASHBOARD ASSET EXPORT
# ============================================================

import joblib
import os


# ============================================================
# PREPARE FULL VALIDATION OVERVIEW DATA
# ============================================================

overview_data = dashboard_data[
    [
        "TransactionID",
        "Actual_Fraud",
        "Risk_Score",
        "Fraud_Prediction"
    ]
].copy()


# ============================================================
# PREPARE DASHBOARD ASSETS
# ============================================================

dashboard_assets = {

    # Full validation population for overall analytics
    "overview_data":
        overview_data,

    # 5,000 SHAP investigation transactions
    "investigation_data":
        investigation_data,

    # Compact local SHAP explanations
    "shap_dashboard_data":
        shap_dashboard_data,

    # Historical relational evidence
    "graph_connections":
        graph_connections_dashboard
}


assets_path = (
    "/kaggle/working/dashboard_assets.pkl"
)


joblib.dump(
    dashboard_assets,
    assets_path,
    compress=3
)


# ============================================================
# VALIDATE EXPORTED ASSETS
# ============================================================

loaded_assets = joblib.load(
    assets_path
)


required_assets = {
    "overview_data",
    "investigation_data",
    "shap_dashboard_data",
    "graph_connections"
}


assert required_assets.issubset(
    loaded_assets.keys()
)


# Full validation population
assert (
    len(
        loaded_assets["overview_data"]
    )
    == len(valid_df)
)


# Investigation sample
assert (
    loaded_assets[
        "investigation_data"
    ]["TransactionID"].nunique()
    == 5000
)


# SHAP sample
assert (
    loaded_assets[
        "shap_dashboard_data"
    ]["TransactionID"].nunique()
    == 5000
)


# Verify investigation IDs belong to validation data
assert loaded_assets[
    "investigation_data"
]["TransactionID"].isin(
    loaded_assets[
        "overview_data"
    ]["TransactionID"]
).all()


graph_source_count = (
    loaded_assets[
        "graph_connections"
    ]["Source_Transaction"].nunique()

    if len(
        loaded_assets[
            "graph_connections"
        ]
    ) > 0

    else 0
)


# ============================================================
# RESULTS
# ============================================================

print(
    "Dashboard assets exported successfully."
)

print(
    "File:",
    assets_path
)

print(
    "File exists:",
    os.path.exists(
        assets_path
    )
)


print("\nExported dashboard data:")

print(
    "Full validation transactions:",
    len(
        loaded_assets[
            "overview_data"
        ]
    )
)

print(
    "Investigation transactions:",
    len(
        loaded_assets[
            "investigation_data"
        ]
    )
)

print(
    "SHAP explanation rows:",
    len(
        loaded_assets[
            "shap_dashboard_data"
        ]
    )
)

print(
    "Transactions with relational networks:",
    graph_source_count
)

print(
    "Historical connection records:",
    len(
        loaded_assets[
            "graph_connections"
        ]
    )
)

print(
    "Asset file size:",
    f"{os.path.getsize(assets_path) / (1024 ** 2):.2f} MB"
)

## 15.8 MAKĀMIN Interactive Fraud Analytics & Investigation Dashboard

The final MAKĀMIN dashboard is organized as a two-level analytical workflow.

The first level provides an overall fraud analysis across the complete chronological validation set. It summarizes transaction volume, known fraud prevalence, model alerts, detected fraud cases, and the distribution of Enhanced XGBoost risk scores.

The second level allows analysts to drill down into one of the 5,000 validation transactions for which local SHAP explanations were computed. For each selected transaction, MAKĀMIN presents its risk assessment, transaction profile, strongest SHAP contributions, and historical relational neighborhood from the previous 24 hours.

Historical relationships are identified through shared card, address, email-domain, or device information. Network statistics use the complete available historical neighborhood, while the interactive graph displays up to 150 connected transactions to maintain readability and responsiveness.

The resulting workflow follows:

**Overall Analysis → Transaction Selection → Risk Assessment → Explanation → Relational Investigation**

In [ ]:
# ============================================================
# 15.8 CREATE PREMIUM MAKĀMIN STREAMLIT DASHBOARD
# ============================================================

import os


app_code = r'''
# ============================================================
# MAKĀMIN | مكامن
# FRAUD ANALYTICS & INVESTIGATION DASHBOARD
# ============================================================

import os
import html
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
import plotly.graph_objects as go

from pyvis.network import Network


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="MAKĀMIN | Fraud Investigation",
    page_icon="◈",
    layout="wide",
    initial_sidebar_state="collapsed"
)


# ============================================================
# PREMIUM CSS
# ============================================================

st.markdown("""
<style>

:root {
    --bg: #070A12;
    --panel: #0D1220;
    --panel2: #111827;
    --purple: #8B5CF6;
    --purple2: #A78BFA;
    --cyan: #38BDF8;
    --green: #34D399;
    --red: #FB7185;
    --amber: #FBBF24;
    --text: #F8FAFC;
    --muted: #94A3B8;
}

.stApp {
    background:
        radial-gradient(circle at 12% 4%,
            rgba(139,92,246,.16), transparent 26%),
        radial-gradient(circle at 88% 10%,
            rgba(56,189,248,.09), transparent 24%),
        linear-gradient(180deg, #070A12 0%, #080C15 100%);
    color: var(--text);
}

.block-container {
    max-width: 1480px;
    padding-top: 2rem;
    padding-bottom: 4rem;
}

/* Hide Streamlit chrome */
#MainMenu {visibility: hidden;}
footer {visibility: hidden;}


/* ============================================================
   HERO
   ============================================================ */

.hero {
    position: relative;
    overflow: hidden;
    padding: 42px 44px;
    border-radius: 28px;
    margin-bottom: 34px;

    background:
        linear-gradient(135deg,
            rgba(139,92,246,.18),
            rgba(56,189,248,.07));

    border: 1px solid rgba(167,139,250,.22);

    box-shadow:
        0 24px 70px rgba(0,0,0,.35),
        inset 0 1px 0 rgba(255,255,255,.05);
}

.hero:after {
    content: "";
    position: absolute;
    width: 280px;
    height: 280px;
    right: -80px;
    top: -100px;
    border-radius: 50%;
    background: rgba(139,92,246,.12);
    filter: blur(10px);
}

.hero-badge {
    display: inline-block;
    padding: 7px 13px;
    border-radius: 999px;
    background: rgba(139,92,246,.14);
    border: 1px solid rgba(167,139,250,.28);
    color: #C4B5FD;
    font-size: 11px;
    font-weight: 800;
    letter-spacing: 1.5px;
    margin-bottom: 17px;
}

.hero-title {
    font-size: 48px;
    line-height: 1.05;
    font-weight: 850;
    color: #FFFFFF;
    letter-spacing: -1.5px;
}

.hero-tagline {
    margin-top: 9px;
    font-size: 19px;
    color: #C4B5FD;
    font-weight: 700;
}

.hero-text {
    margin-top: 17px;
    max-width: 900px;
    color: #CBD5E1;
    font-size: 15.5px;
    line-height: 1.8;
}

.hero-flow {
    margin-top: 22px;
    color: #94A3B8;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: .4px;
}


/* ============================================================
   SECTION HEADERS
   ============================================================ */

.section-kicker {
    color: #A78BFA;
    font-size: 11px;
    font-weight: 850;
    letter-spacing: 1.6px;
    margin-bottom: 6px;
}

.section-title {
    color: #F8FAFC;
    font-size: 29px;
    font-weight: 850;
    letter-spacing: -.4px;
    margin-bottom: 8px;
}

.section-desc {
    color: #94A3B8;
    line-height: 1.7;
    margin-bottom: 24px;
    max-width: 1000px;
}


/* ============================================================
   KPI CARDS
   ============================================================ */

.kpi-card {
    min-height: 142px;
    padding: 20px 20px;
    border-radius: 19px;

    background:
        linear-gradient(145deg,
            rgba(255,255,255,.065),
            rgba(255,255,255,.025));

    border: 1px solid rgba(255,255,255,.085);

    box-shadow:
        0 14px 35px rgba(0,0,0,.20),
        inset 0 1px 0 rgba(255,255,255,.035);

    transition: .2s ease;
}

.kpi-card:hover {
    transform: translateY(-2px);
    border-color: rgba(167,139,250,.25);
}

.kpi-icon {
    font-size: 22px;
    margin-bottom: 12px;
}

.kpi-label {
    color: #94A3B8;
    font-size: 10.5px;
    font-weight: 850;
    letter-spacing: .9px;
    margin-bottom: 7px;
}

.kpi-value {
    color: #F8FAFC;
    font-size: 26px;
    font-weight: 850;
}


/* ============================================================
   PROFILE
   ============================================================ */

.profile-card {
    min-height: 94px;
    padding: 16px 17px;
    border-radius: 15px;

    background: rgba(255,255,255,.035);
    border: 1px solid rgba(255,255,255,.07);
}

.profile-label {
    color: #94A3B8;
    font-size: 10px;
    font-weight: 850;
    letter-spacing: .9px;
    margin-bottom: 8px;
}

.profile-value {
    color: #F8FAFC;
    font-size: 15px;
    font-weight: 750;
    word-break: break-word;
}


/* ============================================================
   SHAP DIRECTION CARDS
   ============================================================ */

.risk-up {
    min-height: 145px;
    padding: 21px 23px;
    border-radius: 18px;

    background:
        linear-gradient(135deg,
            rgba(251,113,133,.14),
            rgba(251,113,133,.035));

    border: 1px solid rgba(251,113,133,.28);
}

.risk-down {
    min-height: 145px;
    padding: 21px 23px;
    border-radius: 18px;

    background:
        linear-gradient(135deg,
            rgba(52,211,153,.14),
            rgba(52,211,153,.035));

    border: 1px solid rgba(52,211,153,.28);
}

.direction-label {
    font-size: 11px;
    font-weight: 850;
    letter-spacing: 1.1px;
    margin-bottom: 13px;
}

.up-label { color: #FDA4AF; }
.down-label { color: #6EE7B7; }

.direction-feature {
    color: #FFFFFF;
    font-size: 20px;
    font-weight: 850;
    margin-bottom: 5px;
}

.direction-value {
    font-size: 17px;
    font-weight: 850;
}

.up-value { color: #FB7185; }
.down-value { color: #34D399; }

.direction-caption {
    color: #94A3B8;
    font-size: 12px;
    margin-top: 8px;
}


/* ============================================================
   INFO CARDS
   ============================================================ */

.explain-card {
    padding: 19px 21px;
    border-radius: 16px;

    background: rgba(255,255,255,.035);
    border: 1px solid rgba(255,255,255,.075);

    color: #CBD5E1;
    line-height: 1.75;

    margin-top: 12px;
    margin-bottom: 16px;
}

.legend-box {
    padding: 12px 14px;
    border-radius: 12px;

    background: rgba(255,255,255,.035);
    border: 1px solid rgba(255,255,255,.07);

    text-align: center;
    color: #CBD5E1;
}

.soft-divider {
    height: 1px;

    background:
        linear-gradient(
            90deg,
            transparent,
            rgba(167,139,250,.38),
            transparent
        );

    margin: 46px 0;
}


/* ============================================================
   SELECT BOX
   ============================================================ */

div[data-baseweb="select"] > div {
    background-color: #101624;
    border-color: rgba(167,139,250,.20);
}


/* ============================================================
   FOOTER
   ============================================================ */

.footer-box {
    margin-top: 50px;
    padding: 28px;
    border-radius: 20px;

    background:
        linear-gradient(135deg,
            rgba(139,92,246,.07),
            rgba(56,189,248,.03));

    border: 1px solid rgba(255,255,255,.07);

    text-align: center;
    color: #94A3B8;
    line-height: 1.9;
}

.footer-brand {
    color: #FFFFFF;
    font-size: 22px;
    font-weight: 850;
}

.footer-tagline {
    color: #A78BFA;
    font-weight: 750;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD DATA
# ============================================================

BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

ASSETS_PATH = os.path.join(
    BASE_DIR,
    "dashboard_assets.pkl"
)


@st.cache_resource
def load_dashboard_assets():
    return joblib.load(ASSETS_PATH)


assets = load_dashboard_assets()


overview_data = assets[
    "overview_data"
].copy()

investigation_data = assets[
    "investigation_data"
].copy()

shap_dashboard_data = assets[
    "shap_dashboard_data"
].copy()

graph_connections = assets[
    "graph_connections"
].copy()


overview_data["TransactionID"] = (
    overview_data["TransactionID"].astype(int)
)

investigation_data["TransactionID"] = (
    investigation_data["TransactionID"].astype(int)
)

shap_dashboard_data["TransactionID"] = (
    shap_dashboard_data["TransactionID"].astype(int)
)


if len(graph_connections) > 0:

    graph_connections["Source_Transaction"] = (
        graph_connections[
            "Source_Transaction"
        ].astype(int)
    )

    graph_connections["Related_Transaction"] = (
        graph_connections[
            "Related_Transaction"
        ].astype(int)
    )

    graph_connections = (
        graph_connections
        .set_index(
            "Source_Transaction",
            drop=False
        )
        .sort_index()
    )


# ============================================================
# HELPERS
# ============================================================

def kpi_card(icon, label, value):

    st.markdown(
        f"""
<div class="kpi-card">
<div class="kpi-icon">{icon}</div>
<div class="kpi-label">{label}</div>
<div class="kpi-value">{value}</div>
</div>
""",
        unsafe_allow_html=True
    )


def profile_card(label, value):

    st.markdown(
        f"""
<div class="profile-card">
<div class="profile-label">{label}</div>
<div class="profile-value">{html.escape(str(value))}</div>
</div>
""",
        unsafe_allow_html=True
    )


def clean_value(value):

    if pd.isna(value):
        return "Unavailable"

    return str(value)


# ============================================================
# HERO
# ============================================================

st.markdown(
    """
<div class="hero">
<div class="hero-badge">FRAUD INTELLIGENCE · EXPLAINABLE AI · RELATIONAL ANALYSIS</div>
<div class="hero-title">MAKĀMIN | مكامن</div>
<div class="hero-tagline">Uncovering Hidden Fraud Patterns</div>
<div class="hero-text">
A fraud analytics and investigation workspace that combines
Enhanced XGBoost risk scoring, local SHAP explanations, and
historical relational evidence to move from population-level
monitoring to individual transaction investigation.
</div>
<div class="hero-flow">
OVERVIEW &nbsp;→&nbsp; DETECT &nbsp;→&nbsp; EXPLAIN
&nbsp;→&nbsp; INVESTIGATE &nbsp;→&nbsp; VISUALIZE
</div>
</div>
""",
    unsafe_allow_html=True
)


# ============================================================
# 01 OVERVIEW
# ============================================================

st.markdown(
    '<div class="section-kicker">01 · OVERALL ANALYSIS</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-title">Fraud Detection Overview</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-desc">'
    'A population-level view of Enhanced XGBoost predictions '
    'across the complete chronological validation set.'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# METRICS
# ============================================================

total_transactions = len(overview_data)

actual_fraud = int(
    overview_data["Actual_Fraud"].sum()
)

actual_fraud_rate = (
    actual_fraud /
    total_transactions *
    100
)

predicted_fraud = int(
    overview_data["Fraud_Prediction"].sum()
)


true_positive = int(
    (
        (overview_data["Actual_Fraud"] == 1)
        &
        (overview_data["Fraud_Prediction"] == 1)
    ).sum()
)


false_positive = int(
    (
        (overview_data["Actual_Fraud"] == 0)
        &
        (overview_data["Fraud_Prediction"] == 1)
    ).sum()
)


false_negative = int(
    (
        (overview_data["Actual_Fraud"] == 1)
        &
        (overview_data["Fraud_Prediction"] == 0)
    ).sum()
)


true_negative = int(
    (
        (overview_data["Actual_Fraud"] == 0)
        &
        (overview_data["Fraud_Prediction"] == 0)
    ).sum()
)


recall = (
    true_positive /
    (true_positive + false_negative) *
    100
)


precision = (
    true_positive /
    (true_positive + false_positive) *
    100
)


# ============================================================
# KPI ROW
# ============================================================

k1, k2, k3, k4, k5 = st.columns(5)


with k1:
    kpi_card(
        "◈",
        "VALIDATION TRANSACTIONS",
        f"{total_transactions:,}"
    )


with k2:
    kpi_card(
        "◆",
        "KNOWN FRAUD RATE",
        f"{actual_fraud_rate:.2f}%"
    )


with k3:
    kpi_card(
        "◎",
        "MODEL FRAUD ALERTS",
        f"{predicted_fraud:,}"
    )


with k4:
    kpi_card(
        "✓",
        "FRAUD DETECTED",
        f"{true_positive:,}"
    )


with k5:
    kpi_card(
        "↗",
        "RECALL @ 0.50",
        f"{recall:.2f}%"
    )


st.markdown("<br>", unsafe_allow_html=True)


# ============================================================
# OVERVIEW CHARTS
# ============================================================

left_chart, right_chart = st.columns(
    [1.3, 1]
)


# ------------------------------------------------------------
# RISK DISTRIBUTION
# ------------------------------------------------------------

with left_chart:

    st.markdown(
        "#### Risk Score Distribution"
    )

    fig_risk = go.Figure()


    fig_risk.add_trace(
        go.Histogram(
            x=overview_data.loc[
                overview_data[
                    "Actual_Fraud"
                ] == 0,
                "Risk_Score"
            ],

            name="Known Non-Fraud",
            opacity=.72,
            nbinsx=55,

            marker_color="#38BDF8"
        )
    )


    fig_risk.add_trace(
        go.Histogram(
            x=overview_data.loc[
                overview_data[
                    "Actual_Fraud"
                ] == 1,
                "Risk_Score"
            ],

            name="Known Fraud",
            opacity=.78,
            nbinsx=55,

            marker_color="#FB7185"
        )
    )


    fig_risk.add_vline(
        x=.50,
        line_dash="dash",
        line_color="#FBBF24",
        annotation_text="0.50 threshold",
        annotation_font_color="#FBBF24"
    )


    fig_risk.update_layout(
        barmode="overlay",
        height=410,

        template="plotly_dark",

        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(255,255,255,.015)",

        xaxis_title="Enhanced XGBoost Risk Score",
        yaxis_title="Transactions",

        legend=dict(
            orientation="h",
            y=1.12
        ),

        margin=dict(
            l=20,
            r=20,
            t=40,
            b=20
        )
    )


    st.plotly_chart(
        fig_risk,
        use_container_width=True
    )


# ------------------------------------------------------------
# OUTCOMES
# ------------------------------------------------------------

with right_chart:

    st.markdown(
        "#### Reference-Threshold Outcomes"
    )


    outcome_labels = [
        "Fraud Detected",
        "Fraud Missed",
        "False Alerts"
    ]


    outcome_values = [
        true_positive,
        false_negative,
        false_positive
    ]


    outcome_colors = [
        "#34D399",
        "#FB7185",
        "#FBBF24"
    ]


    fig_outcomes = go.Figure(
        go.Bar(
            x=outcome_labels,
            y=outcome_values,

            marker_color=outcome_colors,

            text=[
                f"{x:,}"
                for x in outcome_values
            ],

            textposition="outside"
        )
    )


    fig_outcomes.update_layout(
        height=410,

        template="plotly_dark",

        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(255,255,255,.015)",

        yaxis_title="Transactions",

        showlegend=False,

        margin=dict(
            l=20,
            r=20,
            t=40,
            b=20
        )
    )


    st.plotly_chart(
        fig_outcomes,
        use_container_width=True
    )


# ============================================================
# EVALUATION CONTEXT
# ============================================================

st.markdown(
    f"""
<div class="explain-card">
<b>Evaluation snapshot</b><br><br>
At the reference threshold of <b>0.50</b>, Enhanced XGBoost
detected <b>{true_positive:,}</b> known fraud transactions,
with <b>{recall:.2f}% recall</b> and
<b>{precision:.2f}% precision</b>.
The threshold is retained as a reference point for model
evaluation and dashboard demonstration; it is not presented
as an optimized production threshold.
</div>
""",
    unsafe_allow_html=True
)


st.markdown(
    '<div class="soft-divider"></div>',
    unsafe_allow_html=True
)


# ============================================================
# 02 CASE INVESTIGATION
# ============================================================

st.markdown(
    '<div class="section-kicker">02 · CASE INVESTIGATION</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-title">Investigate a Transaction</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-desc">'
    'Drill down into one of the 5,000 validation transactions '
    'included in the local SHAP investigation sample.'
    '</div>',
    unsafe_allow_html=True
)


available_transactions = (
    investigation_data[
        "TransactionID"
    ].astype(int).tolist()
)


selected_transaction_id = st.selectbox(
    "Select Transaction ID",
    available_transactions,
    index=0
)


selected_row = (
    investigation_data.loc[
        investigation_data[
            "TransactionID"
        ] == selected_transaction_id
    ].iloc[0]
)


risk_score = float(
    selected_row["Risk_Score"]
)

risk_percent = (
    risk_score * 100
)

prediction = int(
    selected_row["Fraud_Prediction"]
)

actual_label = int(
    selected_row["Actual_Fraud"]
)


prediction_text = (
    "Fraud Alert"
    if prediction == 1
    else "No Fraud Alert"
)


actual_text = (
    "Fraud"
    if actual_label == 1
    else "Non-Fraud"
)


# ============================================================
# RISK ASSESSMENT
# ============================================================

st.markdown(
    "### Risk Assessment"
)


risk_left, risk_right = st.columns(
    [1.15, 1.85]
)


# ------------------------------------------------------------
# GAUGE
# ------------------------------------------------------------

with risk_left:

    fig_gauge = go.Figure(
        go.Indicator(
            mode="gauge+number",

            value=risk_percent,

            number={
                "suffix": "%",
                "font": {
                    "size": 38,
                    "color": "#F8FAFC"
                }
            },

            title={
                "text": "Enhanced XGBoost Risk Score",
                "font": {
                    "size": 15,
                    "color": "#CBD5E1"
                }
            },

            gauge={
                "axis": {
                    "range": [0, 100],
                    "tickcolor": "#64748B"
                },

                "bar": {
                    "color": (
                        "#FB7185"
                        if risk_percent >= 50
                        else "#38BDF8"
                    )
                },

                "bgcolor": "#111827",

                "borderwidth": 0,

                "steps": [
                    {
                        "range": [0, 50],
                        "color": "rgba(52,211,153,.10)"
                    },
                    {
                        "range": [50, 100],
                        "color": "rgba(251,113,133,.12)"
                    }
                ],

                "threshold": {
                    "line": {
                        "color": "#FBBF24",
                        "width": 4
                    },
                    "thickness": .8,
                    "value": 50
                }
            }
        )
    )


    fig_gauge.update_layout(
        height=290,
        paper_bgcolor="rgba(0,0,0,0)",
        margin=dict(
            l=25,
            r=25,
            t=45,
            b=10
        )
    )


    st.plotly_chart(
        fig_gauge,
        use_container_width=True
    )


# ------------------------------------------------------------
# RISK CARDS
# ------------------------------------------------------------

with risk_right:

    r1, r2, r3 = st.columns(3)


    with r1:
        kpi_card(
            "ID",
            "TRANSACTION",
            selected_transaction_id
        )


    with r2:
        kpi_card(
            "AI",
            "MODEL DECISION",
            prediction_text
        )


    with r3:
        kpi_card(
            "GT",
            "KNOWN LABEL",
            actual_text
        )


    st.markdown("<br>", unsafe_allow_html=True)

    st.markdown(
        """
<div class="explain-card">
The yellow marker on the gauge represents the
<b>0.50 reference threshold</b>.
The known label is shown because this is validation data;
it would not be available when scoring a new transaction
in deployment.
</div>
""",
        unsafe_allow_html=True
    )


# ============================================================
# PROFILE
# ============================================================

st.markdown(
    "### Transaction Profile"
)


p1, p2, p3, p4, p5 = st.columns(5)


with p1:
    profile_card(
        "TRANSACTION AMOUNT",
        clean_value(
            selected_row["TransactionAmt"]
        )
    )


with p2:
    profile_card(
        "CARD",
        clean_value(
            selected_row["card1"]
        )
    )


with p3:
    profile_card(
        "ADDRESS",
        clean_value(
            selected_row["addr1"]
        )
    )


with p4:
    profile_card(
        "EMAIL DOMAIN",
        clean_value(
            selected_row["P_emaildomain"]
        )
    )


with p5:
    profile_card(
        "DEVICE",
        clean_value(
            selected_row["DeviceInfo"]
        )
    )


st.markdown(
    '<div class="soft-divider"></div>',
    unsafe_allow_html=True
)


# ============================================================
# 03 EXPLAINABILITY
# ============================================================

st.markdown(
    '<div class="section-kicker">03 · EXPLAINABILITY</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-title">What Drove the Risk Score?</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-desc">'
    'Local SHAP values reveal which features pushed the model '
    'toward higher fraud risk and which features reduced it.'
    '</div>',
    unsafe_allow_html=True
)


transaction_shap = (
    shap_dashboard_data.loc[
        shap_dashboard_data[
            "TransactionID"
        ] == selected_transaction_id
    ].copy()
)


transaction_shap[
    "Abs_SHAP"
] = transaction_shap[
    "SHAP_Value"
].abs()


# ============================================================
# STRONGEST INCREASE / DECREASE
# ============================================================

positive_shap = (
    transaction_shap.loc[
        transaction_shap[
            "SHAP_Value"
        ] > 0
    ]
    .sort_values(
        "SHAP_Value",
        ascending=False
    )
)


negative_shap = (
    transaction_shap.loc[
        transaction_shap[
            "SHAP_Value"
        ] < 0
    ]
    .sort_values(
        "SHAP_Value",
        ascending=True
    )
)


if len(positive_shap) > 0:

    strongest_up = positive_shap.iloc[0]

    up_feature = html.escape(
        str(
            strongest_up[
                "Feature"
            ]
        )
    )

    up_value = float(
        strongest_up[
            "SHAP_Value"
        ]
    )

else:

    up_feature = "None"
    up_value = 0.0


if len(negative_shap) > 0:

    strongest_down = negative_shap.iloc[0]

    down_feature = html.escape(
        str(
            strongest_down[
                "Feature"
            ]
        )
    )

    down_value = float(
        strongest_down[
            "SHAP_Value"
        ]
    )

else:

    down_feature = "None"
    down_value = 0.0


direction_left, direction_right = st.columns(2)


with direction_left:

    st.markdown(
        f"""
<div class="risk-up">
<div class="direction-label up-label">
▲ FRAUD RISK INCREASE
</div>
<div class="direction-feature">
{up_feature}
</div>
<div class="direction-value up-value">
{up_value:+.3f} SHAP
</div>
<div class="direction-caption">
Strongest displayed feature pushing this prediction
toward higher fraud risk.
</div>
</div>
""",
        unsafe_allow_html=True
    )


with direction_right:

    st.markdown(
        f"""
<div class="risk-down">
<div class="direction-label down-label">
▼ FRAUD RISK DECREASE
</div>
<div class="direction-feature">
{down_feature}
</div>
<div class="direction-value down-value">
{down_value:+.3f} SHAP
</div>
<div class="direction-caption">
Strongest displayed feature pushing this prediction
toward lower fraud risk.
</div>
</div>
""",
        unsafe_allow_html=True
    )


st.markdown("<br>", unsafe_allow_html=True)


# ============================================================
# SHAP BAR CHART
# ============================================================

transaction_shap = (
    transaction_shap
    .sort_values(
        "Abs_SHAP",
        ascending=True
    )
)


shap_colors = [
    "#FB7185"
    if value > 0
    else "#34D399"

    for value in transaction_shap[
        "SHAP_Value"
    ]
]


fig_shap = go.Figure(
    go.Bar(
        x=transaction_shap[
            "SHAP_Value"
        ],

        y=transaction_shap[
            "Feature"
        ],

        orientation="h",

        marker_color=shap_colors,

        text=[
            f"{x:+.3f}"
            for x in transaction_shap[
                "SHAP_Value"
            ]
        ],

        textposition="outside"
    )
)


fig_shap.add_vline(
    x=0,
    line_color="#64748B",
    line_width=1.2
)


fig_shap.update_layout(
    height=490,

    template="plotly_dark",

    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(255,255,255,.015)",

    xaxis_title="SHAP Contribution",
    yaxis_title="",

    showlegend=False,

    margin=dict(
        l=20,
        r=60,
        t=25,
        b=20
    )
)


st.plotly_chart(
    fig_shap,
    use_container_width=True
)


st.markdown(
    """
<div class="explain-card">
<span style="color:#FB7185"><b>Red</b></span>
features push the model toward a higher fraud risk.
&nbsp;&nbsp;
<span style="color:#34D399"><b>Green</b></span>
features push the model toward a lower fraud risk.
<br><br>
SHAP explains the model's local prediction; it does not
establish that an individual feature caused fraud.
</div>
""",
    unsafe_allow_html=True
)


st.markdown(
    '<div class="soft-divider"></div>',
    unsafe_allow_html=True
)


# ============================================================
# 04 RELATIONAL INVESTIGATION
# ============================================================

st.markdown(
    '<div class="section-kicker">04 · RELATIONAL INVESTIGATION</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-title">Historical Transaction Network</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="section-desc">'
    'Explore earlier transactions from the previous 24 hours '
    'that share a card, address, email domain, or device with '
    'the selected transaction.'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# SELECT CONNECTIONS
# ============================================================

if (
    len(graph_connections) > 0
    and
    selected_transaction_id
    in graph_connections.index
):

    selected_connections = (
        graph_connections.loc[
            [selected_transaction_id]
        ].copy()
    )

else:

    selected_connections = pd.DataFrame(
        columns=[
            "Source_Transaction",
            "Related_Transaction",
            "Connection_Type",
            "Actual_Fraud"
        ]
    )


# ============================================================
# CONNECTION ANALYSIS
# ============================================================

if len(selected_connections) > 0:

    total_connections = len(
        selected_connections
    )

    fraud_connections = int(
        selected_connections[
            "Actual_Fraud"
        ].sum()
    )

    nonfraud_connections = (
        total_connections
        - fraud_connections
    )

    fraud_share = (
        fraud_connections /
        total_connections *
        100
    )


    n1, n2, n3, n4 = st.columns(4)


    with n1:
        kpi_card(
            "◎",
            "HISTORICAL CONNECTIONS",
            f"{total_connections:,}"
        )


    with n2:
        kpi_card(
            "●",
            "KNOWN FRAUD-LABELED",
            f"{fraud_connections:,}"
        )


    with n3:
        kpi_card(
            "●",
            "KNOWN NON-FRAUD",
            f"{nonfraud_connections:,}"
        )


    with n4:
        kpi_card(
            "%",
            "FRAUD-LABELED SHARE",
            f"{fraud_share:.2f}%"
        )


    st.markdown("<br>", unsafe_allow_html=True)


    # ========================================================
    # CONNECTION TYPES
    # ========================================================

    connection_types = (
        selected_connections[
            "Connection_Type"
        ]
        .str.split(", ")
        .explode()
        .value_counts()
    )


    st.markdown(
        "#### Shared-Entity Evidence"
    )


    fig_types = go.Figure(
        go.Bar(
            x=connection_types.index,
            y=connection_types.values,

            marker_color=[
                "#8B5CF6",
                "#38BDF8",
                "#34D399",
                "#FBBF24"
            ][:len(connection_types)],

            text=connection_types.values,

            textposition="outside"
        )
    )


    fig_types.update_layout(
        height=350,

        template="plotly_dark",

        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(255,255,255,.015)",

        xaxis_title="Shared Entity",
        yaxis_title="Historical Connections",

        showlegend=False,

        margin=dict(
            l=20,
            r=20,
            t=25,
            b=20
        )
    )


    st.plotly_chart(
        fig_types,
        use_container_width=True
    )


    # ========================================================
    # GRAPH DISPLAY SUBSET
    # ========================================================

    MAX_GRAPH_NODES = 150


    fraud_neighbors = (
        selected_connections.loc[
            selected_connections[
                "Actual_Fraud"
            ] == 1
        ].copy()
    )


    nonfraud_neighbors = (
        selected_connections.loc[
            selected_connections[
                "Actual_Fraud"
            ] == 0
        ].copy()
    )


    if len(fraud_neighbors) >= MAX_GRAPH_NODES:

        display_connections = (
            fraud_neighbors.sample(
                n=MAX_GRAPH_NODES,
                random_state=42
            )
        )

    else:

        remaining_slots = (
            MAX_GRAPH_NODES
            - len(fraud_neighbors)
        )


        if len(
            nonfraud_neighbors
        ) > remaining_slots:

            sampled_nonfraud = (
                nonfraud_neighbors.sample(
                    n=remaining_slots,
                    random_state=42
                )
            )

        else:

            sampled_nonfraud = (
                nonfraud_neighbors
            )


        display_connections = pd.concat(
            [
                fraud_neighbors,
                sampled_nonfraud
            ],
            ignore_index=True
        )


    displayed_connections = len(
        display_connections
    )


    # ========================================================
    # NETWORK
    # ========================================================

    st.markdown(
        "#### Interactive Relationship Map"
    )


    net = Network(
        height="660px",
        width="100%",
        bgcolor="#080C15",
        font_color="#F8FAFC",
        directed=False,
        cdn_resources="in_line"
    )


    net.add_node(
        selected_transaction_id,

        label=(
            f"SELECTED\n"
            f"{selected_transaction_id}"
        ),

        title=(
            f"Selected transaction"
            f"<br>Risk score: {risk_percent:.2f}%"
            f"<br>Model decision: {prediction_text}"
        ),

        color="#8B5CF6",
        size=42,
        borderWidth=4
    )


    for _, row in display_connections.iterrows():

        related_id = int(
            row["Related_Transaction"]
        )

        related_fraud = int(
            row["Actual_Fraud"]
        )

        connection_type = str(
            row["Connection_Type"]
        )


        if related_fraud == 1:

            node_color = "#FB7185"
            node_size = 20

        else:

            node_color = "#34D399"
            node_size = 11


        net.add_node(
            related_id,

            label=str(
                related_id
            ),

            title=(
                f"Transaction: {related_id}"
                f"<br>Known fraud label: {related_fraud}"
                f"<br>Shared entity: {connection_type}"
            ),

            color=node_color,
            size=node_size
        )


        net.add_edge(
            selected_transaction_id,
            related_id,

            title=(
                f"Shared: {connection_type}"
            ),

            color="#334155"
        )


    net.set_options("""
    {
      "nodes": {
        "shape": "dot",
        "font": {
          "size": 10,
          "color": "#E2E8F0"
        },
        "borderWidth": 1
      },

      "edges": {
        "width": 0.8,
        "color": {
          "color": "#334155",
          "highlight": "#A78BFA"
        },
        "smooth": {
          "type": "continuous"
        }
      },

      "interaction": {
        "hover": true,
        "tooltipDelay": 70,
        "navigationButtons": true,
        "keyboard": true
      },

      "physics": {
        "enabled": true,

        "barnesHut": {
          "gravitationalConstant": -7000,
          "centralGravity": 0.13,
          "springLength": 110,
          "springConstant": 0.025,
          "damping": 0.42
        },

        "stabilization": {
          "iterations": 200
        }
      }
    }
    """)


    graph_html_path = os.path.join(
        BASE_DIR,
        "makamin_network_temp.html"
    )


    net.save_graph(
        graph_html_path
    )


    with open(
        graph_html_path,
        "r",
        encoding="utf-8"
    ) as file:

        graph_html = file.read()


    components.html(
        graph_html,
        height=660,
        scrolling=False
    )


    # ========================================================
    # LEGEND
    # ========================================================

    l1, l2, l3 = st.columns(3)


    with l1:
        st.markdown(
            '<div class="legend-box">'
            '🟣 <b>Selected Transaction</b>'
            '</div>',
            unsafe_allow_html=True
        )


    with l2:
        st.markdown(
            '<div class="legend-box">'
            '🔴 <b>Known Fraud-Labeled</b>'
            '</div>',
            unsafe_allow_html=True
        )


    with l3:
        st.markdown(
            '<div class="legend-box">'
            '🟢 <b>Known Non-Fraud</b>'
            '</div>',
            unsafe_allow_html=True
        )


    st.markdown(
        f"""
<div class="explain-card">
<b>Relationship context</b><br><br>
The selected transaction has
<b>{total_connections:,}</b> historical connections in its
complete previous-24-hour neighborhood, including
<b>{fraud_connections:,}</b> connections to transactions
with known fraud labels.
<br><br>
For readability, the interactive map displays
<b>{displayed_connections:,}</b> connections. Fraud-labeled
neighbors are retained first when the neighborhood exceeds
the display limit, while dashboard statistics continue to use
the complete neighborhood.
<br><br>
Shared entities are investigative signals only and do not
independently establish coordinated fraud.
</div>
""",
        unsafe_allow_html=True
    )


    # ========================================================
    # EXPANDERS
    # ========================================================

    with st.expander(
        "View displayed relational evidence"
    ):

        st.dataframe(
            display_connections[
                [
                    "Related_Transaction",
                    "Connection_Type",
                    "Actual_Fraud"
                ]
            ],

            use_container_width=True,
            hide_index=True
        )


    with st.expander(
        "View complete neighborhood summary"
    ):

        summary_data = pd.DataFrame({

            "Metric": [
                "Historical connections",
                "Known fraud-labeled connections",
                "Known non-fraud connections",
                "Fraud-labeled share",
                "Connections displayed"
            ],

            "Value": [
                f"{total_connections:,}",
                f"{fraud_connections:,}",
                f"{nonfraud_connections:,}",
                f"{fraud_share:.2f}%",
                f"{displayed_connections:,}"
            ]
        })


        st.dataframe(
            summary_data,
            use_container_width=True,
            hide_index=True
        )


# ============================================================
# NO CONNECTIONS
# ============================================================

else:

    st.markdown(
        """
<div class="explain-card">
<b>No historical relational evidence found.</b>
<br><br>
No earlier transaction in the previous 24 hours shared the
investigated card, address, email domain, or device.
The absence of a historical neighborhood does not determine
whether the selected transaction is fraudulent.
</div>
""",
        unsafe_allow_html=True
    )


# ============================================================
# CONTEXT / LIMITATIONS
# ============================================================

st.markdown(
    '<div class="soft-divider"></div>',
    unsafe_allow_html=True
)


st.markdown(
    """
<div class="explain-card">
<b>Evaluation context</b><br><br>
Population-level analytics use the complete chronological
validation set of <b>118,108 transactions</b>.
Transaction-level investigation is available for
<b>5,000 validation transactions</b> with local SHAP
explanations.
<br><br>
Known fraud labels are displayed for evaluation and
demonstration only. They would not be available when scoring
a new transaction in deployment. Historical relationships use
only earlier transactions from the previous 24 hours.
</div>
""",
    unsafe_allow_html=True
)


# ============================================================
# FOOTER
# ============================================================

st.markdown(
    """
<div class="footer-box">
<div class="footer-brand">MAKĀMIN | مكامن</div>
<div class="footer-tagline">Uncovering Hidden Fraud Patterns</div>
<br>
Fraud Analytics &amp; Investigation Workspace
<br>
Enhanced XGBoost &nbsp;•&nbsp;
Explainable AI (SHAP) &nbsp;•&nbsp;
Historical Relational Analysis
</div>
""",
    unsafe_allow_html=True
)

'''


# ============================================================
# WRITE APP
# ============================================================

app_path = "/kaggle/working/app.py"


with open(
    app_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(app_code)


# ============================================================
# VALIDATE FILE
# ============================================================

assert os.path.exists(app_path)
assert os.path.getsize(app_path) > 0


print(
    "✨ Premium MAKĀMIN dashboard created successfully."
)

print(
    "File:",
    app_path
)

print(
    "File size:",
    os.path.getsize(app_path),
    "bytes"
)

## 15.9 Streamlit Application Validation

Before launching the dashboard, the generated Streamlit application is compiled to verify that the Python file contains no syntax errors.

This validation step helps detect application-level coding issues before starting the Streamlit server.

In [ ]:
# ============================================================
# 15.9 STREAMLIT APPLICATION VALIDATION
# ============================================================

!python -m py_compile /kaggle/working/app.py

print("Streamlit application syntax validated successfully.")

## 15.10 Dashboard Launch

After validating the application syntax, the Streamlit server is launched on port **8501**.

The application is started as a background process so that the notebook can continue executing while the dashboard remains available. A short server check is then performed to confirm that the Streamlit process started successfully.

In [ ]:
!pkill -f "streamlit run" || true

In [ ]:
# ============================================================
# 15.10 DASHBOARD LAUNCH
# ============================================================

import subprocess
import time

# Start Streamlit as a background process
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/kaggle/working/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give the server a few seconds to start
time.sleep(5)

# Check whether the process is still running
if streamlit_process.poll() is None:
    print("Streamlit dashboard started successfully.")
    print("Process ID:", streamlit_process.pid)
    print("Port: 8501")
else:
    stdout, stderr = streamlit_process.communicate()

    print("Streamlit failed to start.")
    print(stderr.decode("utf-8"))

## 15.11 External Dashboard Access

A temporary secure tunnel is created to provide external access to the MAKĀMIN Fraud Intelligence Dashboard during the project demonstration.

The external URL is temporary and remains available only while the current Kaggle session and Streamlit application are running.

In [ ]:
# ============================================================
# 15.11 CREATE EXTERNAL DASHBOARD ACCESS
# ============================================================

import os
import re
import time
import subprocess
import urllib.request

cloudflared_path = "/kaggle/working/cloudflared"
cloudflared_log = "/kaggle/working/cloudflared.log"


# Download cloudflared if it is not already available
if not os.path.exists(cloudflared_path):

    print("Downloading Cloudflare tunnel client...")

    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_path
    )

    os.chmod(
        cloudflared_path,
        0o755
    )


# Stop any previous Cloudflare tunnel
os.system(
    'pkill -f "cloudflared tunnel" >/dev/null 2>&1 || true'
)

time.sleep(1)


# Clear previous log
if os.path.exists(cloudflared_log):
    os.remove(cloudflared_log)


# Start a new temporary tunnel
log_file = open(
    cloudflared_log,
    "w"
)

tunnel_process = subprocess.Popen(
    [
        cloudflared_path,
        "tunnel",
        "--url",
        "http://localhost:8501",
        "--no-autoupdate"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)


public_url = None

print("Creating external MAKĀMIN dashboard link...")


# Wait for Cloudflare to generate the public URL
for _ in range(30):

    time.sleep(1)

    if os.path.exists(cloudflared_log):

        with open(
            cloudflared_log,
            "r"
        ) as file:

            log_text = file.read()

        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            log_text
        )

        if match:

            public_url = match.group(0)
            break


if public_url:

    print("\n" + "=" * 60)
    print("✨ MAKĀMIN | مكامن")
    print("FRAUD INTELLIGENCE DASHBOARD")
    print("=" * 60)

    print("\nExternal Dashboard URL:")
    print(public_url)

    print("\nOpen this link in a new browser tab.")
    print("=" * 60)

else:

    print("The external URL was not detected yet.")
    print("Check the Cloudflare log below:\n")

    if os.path.exists(cloudflared_log):

        with open(
            cloudflared_log,
            "r"
        ) as file:

            print(file.read())

# 16. Final Dashboard Summary

The final **MAKĀMIN Fraud Analytics & Investigation Dashboard** integrates the main outputs of the project into a two-level analytical workflow.

At the population level, MAKĀMIN analyzes the complete chronological validation set of **118,108 transactions**, presenting Enhanced XGBoost risk scores, known validation labels, model alerts, detected fraud cases, and risk-score distributions at the reference threshold of **0.50**.

At the case level, analysts can drill down into **5,000 validation transactions** for which local SHAP explanations were computed. For each selected transaction, the dashboard presents its fraud risk score, model decision, transaction profile, strongest local SHAP contributions, and historical relational evidence.

The corrected historical relational analysis identified previous-24-hour connections for **5,000 out of 5,000 investigation transactions**, producing **3,182,746 historical connection records**. These relationships are based on shared card, address, email-domain, or device information.

To maintain readability and dashboard responsiveness, network statistics use the complete historical neighborhood while the interactive graph displays up to **150 connections** for an individual transaction. When necessary, fraud-labeled neighbors are prioritized in the displayed subset.

The dashboard organizes the investigation workflow as:

**Overall Analysis → Transaction Selection → Risk Assessment → SHAP Explanation → Relational Investigation**

Known fraud labels are displayed only because the dashboard operates on validation data. Historical relationships provide investigation context and should not independently be interpreted as proof of coordinated fraud.

Overall, MAKĀMIN serves as an evaluation and investigation prototype that combines predictive modeling, explainable AI, and historical relational analysis in a single interactive interface.

# 17. Conclusion, Limitations & Future Work

## 17.1 Project Conclusion

This project investigated whether historical relational information could improve fraud detection beyond traditional transaction-level modeling using the **IEEE-CIS Fraud Detection dataset**.

A chronological holdout strategy was used to better reflect a realistic fraud-detection setting, with **472,432 transactions used for training** and **118,108 transactions used for validation**. This preserved the temporal order of transactions and reduced the risk of future information leaking into model development.

Four main predictive configurations were evaluated:

- **Baseline XGBoost**
- **Enhanced XGBoost with Historical Relational Features**
- **Baseline Artificial Neural Network (ANN)**
- **Enhanced ANN with Historical Relational Features**

Among the evaluated models, **Enhanced XGBoost achieved the strongest overall performance**, with:

- **Average Precision: 0.5207**
- **ROC-AUC: 0.9093**

The baseline XGBoost achieved an Average Precision of **0.5189** and ROC-AUC of **0.9078**, showing that the engineered historical relational features provided a **modest improvement** rather than a dramatic predictive gain.

The ANN models produced lower overall performance. The baseline ANN achieved an Average Precision of **0.4509** and ROC-AUC of **0.8612**, while the enhanced ANN achieved an Average Precision of **0.4404** and ROC-AUC of **0.8725**.

These results indicate that the impact of relational features depends on the modeling approach and evaluation metric. In particular, the relational features slightly improved XGBoost and increased ANN ROC-AUC, but did not consistently improve Average Precision across all models.

---

## 17.2 Historical Relational Analysis

To move beyond isolated transaction-level analysis, the project introduced target-free historical relational features based on activity observed during the **previous 24 hours**.

The engineered features captured previous activity associated with:

- Card
- Address
- Email domain
- Device
- Card-address combinations

Only historical information available before each transaction was used when constructing these features.

The final investigation sample contained **5,000 validation transactions**. After validating the transaction alignment between the validation data, SHAP explanations, and dashboard assets, all **5,000 investigation transactions** had at least one historical relationship.
Across this investigation sample, the relational analysis generated:

- **5,000 transactions with historical connections**
- **3,182,746 historical connection records**

Connections were identified when an earlier transaction from the previous 24 hours shared investigated information such as a card, address, email domain, or device.

These relationships should be interpreted as **investigation context rather than direct evidence of coordinated fraud**. A shared identifier may occur for legitimate reasons and does not independently establish that two transactions belong to the same actor or fraud ring.

---

## 17.3 Explainability with SHAP

SHAP was used to improve the interpretability of the Enhanced XGBoost model.

At the global level, SHAP analysis helped identify the features that contributed most strongly to model behavior.

At the local level, SHAP explanations were generated for **5,000 validation transactions**, allowing individual predictions to be investigated in greater detail.

For each selected transaction, the final dashboard distinguishes between:

- **Fraud Risk Increase:** features with positive SHAP contributions that push the model toward a higher fraud-risk prediction.
- **Fraud Risk Decrease:** features with negative SHAP contributions that push the model toward a lower fraud-risk prediction.

This allows analysts to move beyond a single fraud probability and understand which features contributed most strongly to the model's decision for an individual transaction.

SHAP values explain the model's prediction and should not be interpreted as evidence that an individual feature directly caused fraudulent activity.

---

## 17.4 MAKĀMIN Fraud Analytics & Investigation Dashboard

The final project output is the **MAKĀMIN | مكامن Fraud Analytics & Investigation Dashboard**.

MAKĀMIN integrates predictive modeling, explainable AI, and historical relational analysis into a single interactive investigation workflow.

The dashboard operates at two analytical levels.

### Population-Level Analysis

The first level summarizes model behavior across the complete chronological validation set of **118,108 transactions**.

It presents:

- Validation transaction volume
- Known fraud prevalence
- Model fraud alerts
- Detected fraud cases
- Recall at the reference threshold
- Risk-score distributions
- Reference-threshold outcomes

At the reference threshold of **0.50**, Enhanced XGBoost produced:

- **3,014 true-positive fraud detections**
- **1,050 missed fraud transactions**
- **11,108 false-positive alerts**
- **74.16% recall**
- **21.34% precision**

The threshold of 0.50 is retained as a **reference threshold for evaluation and demonstration** and is not presented as an optimized production decision threshold.

### Case-Level Investigation

The second level allows analysts to select one of the **5,000 investigation transactions** and examine:

- Enhanced XGBoost risk score
- Model decision
- Known validation label
- Transaction profile
- Local SHAP explanation
- Strongest fraud-risk-increasing features
- Strongest fraud-risk-decreasing features
- Historical relational connections
- Shared-entity evidence
- Interactive transaction network

Historical network statistics are calculated using the transaction's complete available previous-24-hour neighborhood.

For dashboard readability and responsiveness, the interactive graph displays up to **150 historical connections** for an individual transaction while retaining statistics from the complete available neighborhood.

The final MAKĀMIN workflow can therefore be summarized as:

**Overall Analysis → Transaction Selection → Risk Assessment → Explanation → Relational Investigation → Visualization**

---

## 17.5 Project Limitations

Several limitations should be considered when interpreting the results.

### Relational Features Produced Modest Predictive Improvement

The Enhanced XGBoost model improved over the baseline XGBoost model, but the improvement was relatively small. Therefore, the project does not claim that the engineered relational features fundamentally transformed predictive performance.

Their additional value is also demonstrated through the investigation context they provide around individual transactions.

### Shared Entities Do Not Prove Coordinated Fraud

Transactions connected through the same card, address, email domain, or device are not automatically part of a coordinated fraud ring.

Some shared identifiers, particularly common email domains or shared addresses and devices, may occur naturally in legitimate activity.

The network should therefore support investigation rather than independently determine fraud.

### Validation Labels Are Available Only for Evaluation

The MAKĀMIN dashboard displays known fraud labels because it operates on historical validation data.

In a real deployment, the true fraud label of a newly arriving transaction would not be available at prediction time.

### Reference Threshold

The dashboard uses a fraud probability threshold of **0.50** for consistency with the evaluation results.

This threshold was not optimized as a production operating point. A deployed fraud-detection system would require threshold selection based on business costs, investigation capacity, false-positive tolerance, and the cost of missed fraud.

### Dataset Constraints

The IEEE-CIS dataset contains anonymized features and does not provide complete real-world entity identities or confirmed fraud-ring labels.

As a result, the project evaluates transaction fraud detection and historical relational patterns rather than directly validating confirmed coordinated fraud organizations.

---

## 17.6 Future Work

Several extensions could build on the current MAKĀMIN framework.

### Graph-Based Learning

The historical relational analysis could be extended into richer graph representations in which transactions and shared entities are explicitly modeled as nodes and relationships.

Future experiments could evaluate **Graph Neural Networks (GNNs)** and compare their performance with the current tabular and graph-engineered approaches.

### Richer Temporal Relational Features

Additional historical windows and behavioral features could be investigated, such as:

- Multiple historical time windows
- Transaction velocity
- Repeated entity combinations
- Changes in transaction behavior over time
- Relational frequency and recency
- Neighborhood-level risk statistics constructed without target leakage

### Production-Oriented Threshold Selection

Rather than relying on the 0.50 reference threshold, future work could optimize the operating threshold according to real business requirements.

This could include explicit costs for:

- Missed fraud
- False alerts
- Manual investigation
- Customer friction

### Real-Time Fraud Monitoring

The current system is an evaluation and investigation prototype.

A future production architecture could process newly arriving transactions, update historical entity activity, generate real-time fraud scores, and surface high-risk cases to investigators through MAKĀMIN.

### External and Temporal Validation

Future evaluation should test the system on newly arriving or out-of-time transaction data to measure whether predictive and relational patterns remain stable under changing fraud behavior.

---

## 17.7 Final Takeaway

The project demonstrates that fraud detection can benefit from combining **transaction-level predictive modeling with historical relational context and explainable AI**.

Enhanced XGBoost delivered the strongest predictive performance among the evaluated models, while the engineered relational features provided a modest additional improvement and enabled richer transaction-level investigation.

SHAP made individual model decisions more interpretable, while historical relational analysis provided additional context about how transactions were connected to earlier activity.

The final **MAKĀMIN | مكامن** dashboard brings these components together into a unified analytical workflow:

**Detect the risk. Explain the prediction. Investigate the relationships.**

Rather than treating a fraud score as the final answer, MAKĀMIN provides analysts with additional context for understanding and investigating suspicious transaction activity.